# CMB Tutorial — Companion Notebook

Every code cell from the tutorial in order. Run top to bottom; each cell defines what later cells use.

**Prerequisites:** `numpy`, `scipy`, `matplotlib`, optional `numba`. CAMB is used only in validation cells.

```
pip install numpy scipy matplotlib jupyter
pip install camb  # for validation only
```


## Setup

`np.trapz` was renamed to `np.trapezoid` in numpy 2.0. This shim keeps the rest of the notebook working either way.


In [ ]:
import numpy as np
if not hasattr(np, "trapz"):
    np.trapz = np.trapezoid


# Prerequisites


### Cell 1


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate, interpolate, optimize, special

# Optional: numba speeds up the inner Boltzmann RHS by ~10x.
# Falls back gracefully if not installed.
try:
    from numba import njit
    _jit = njit(cache=False)
except ImportError:
    _jit = lambda f: f

# Default: Planck 2018 best-fit LCDM
params = {
    'omega_b_h2': 0.02237,    # physical baryon density
    'omega_c_h2': 0.1200,     # physical cold dark matter density
    'h':          0.6736,     # H_0 / (100 km/s/Mpc)
    'n_s':        0.9649,     # scalar spectral index
    'A_s':        2.1e-9,     # scalar amplitude at k_pivot = 0.05 Mpc^-1
    'tau_reion':  0.0544,     # reionisation optical depth
    'N_eff':      3.044,      # effective number of relativistic species
    'T_cmb':      2.7255,     # CMB temperature today (K)
    'Y_He':       0.245,      # primordial helium mass fraction
    'k_pivot':    0.05,       # pivot scale (1/Mpc)
    'ell_max':    2500,       # maximum multipole we will compute
}

plt.rcParams.update({'figure.figsize': (7, 4.5), 'font.size': 11})


# Chapter 1 — Background


### Cell: physical constants


In [ ]:
# Physical constants used throughout the tutorial.
c_km_s    = 2.99792458e5             # speed of light (km/s)
k_B       = 1.380649e-23             # Boltzmann constant (J/K)
h_P       = 6.62607015e-34           # Planck constant (J s)
m_e       = 9.1093837015e-31         # electron mass (kg)
m_H       = 1.673575e-27             # hydrogen atom mass (kg)
m_He4     = 6.646479073e-27          # He-4 atom mass (kg)
not4      = m_He4 / m_H              # He/H mass ratio (~3.97, not exactly 4)
sigma_T   = 6.6524587321e-29         # Thomson cross section (m^2)
G         = 6.67430e-11              # gravitational constant (m^3/kg/s^2)
Mpc_in_m  = 3.0856775814913673e22    # 1 Mpc in metres
sigma_SB  = 5.670374419e-8           # Stefan-Boltzmann constant (W/m^2/K^4)


### Cell: `init_background`


In [ ]:
def init_background(params):
    """Convert cosmological parameters to internal `grho_i` densities,
    where grho_i = 8*pi*G*rho_{i,0} in Mpc^-2 (CAMB convention).
    """
    h    = params['h']
    H0   = 100 * h / c_km_s                       # H_0 in 1/Mpc (c = 1)
    grhocrit_h2 = 3 * (100.0 / c_km_s)**2         # 3*(H_100/c)^2 in 1/Mpc^2

    # Photon energy density from the blackbody formula
    T_cmb = params['T_cmb']
    rho_gamma = 4 * sigma_SB / (c_km_s * 1e3)**3 * T_cmb**4   # kg/m^3
    H100_SI       = 100 * 1e3 / Mpc_in_m                       # 1/s
    rho_crit_100  = 3 * H100_SI**2 / (8 * np.pi * G)           # kg/m^3
    omega_gamma   = rho_gamma / rho_crit_100

    # Internal densities: 8*pi*G*rho_{i,0} in Mpc^-2
    grhog       = grhocrit_h2 * omega_gamma                              # photons     (a^-4)
    grhornomass = grhog * 7/8 * (4/11)**(4/3) * params['N_eff']          # massless nu (a^-4)
    grhoc       = grhocrit_h2 * params['omega_c_h2']                     # CDM         (a^-3)
    grhob       = grhocrit_h2 * params['omega_b_h2']                     # baryons     (a^-3)

    # Cosmological constant: Omega_L = 1 - Omega_m - Omega_r (flat universe)
    omega_m = (params['omega_c_h2'] + params['omega_b_h2']) / h**2
    omega_r = omega_gamma * (1 + 7/8 * (4/11)**(4/3) * params['N_eff']) / h**2
    grhov   = grhocrit_h2 * (1 - omega_m - omega_r) * h**2               # Lambda      (const)

    # Two thermodynamic quantities used much later (chapter 2): the
    # Thomson opacity prefactor, and the He/H number-fraction f_He.
    rho_b_SI = params['omega_b_h2'] * rho_crit_100
    n_H_Mpc  = (1 - params['Y_He']) * rho_b_SI / m_H * Mpc_in_m**3
    akthom   = (sigma_T / Mpc_in_m**2) * n_H_Mpc
    f_He     = params['Y_He'] / (not4 * (1 - params['Y_He']))

    return {
        'H0': H0, 'h': h,
        'grhog': grhog, 'grhornomass': grhornomass,
        'grhoc': grhoc, 'grhob': grhob, 'grhov': grhov,
        'akthom': akthom, 'f_He': f_He,
        'Y_He': params['Y_He'], 'T_cmb': T_cmb,
    }


### Cell: Friedmann evaluators


In [ ]:
def total_density_a4(a, bg):
    """Returns 8*pi*G*rho_tot*a^4 -- the polynomial in eq. (above) that
    sits inside the square root of the Friedmann equation."""
    return (bg['grhog'] + bg['grhornomass']
            + (bg['grhoc'] + bg['grhob']) * a
            + bg['grhov'] * a**4)

def dtau_da(a, bg):
    """d(eta)/da = 1/(a^2 H) = sqrt(3 / total_density_a4), in Mpc."""
    return np.sqrt(3.0 / total_density_a4(a, bg))

def hubble(a, bg):
    """Hubble parameter H(a) = sqrt(total_density_a4 / 3) / a^2, in 1/Mpc (c=1)."""
    return np.sqrt(total_density_a4(a, bg) / 3.0) / a**2


### Cell: integrals over the background


In [ ]:
def conformal_time(a, bg):
    """eta(a) = int_0^a da'/(a'^2 H), in Mpc."""
    a = np.atleast_1d(a)
    out = np.array([
        integrate.quad(dtau_da, 0, ai, args=(bg,), limit=100, epsrel=1e-8)[0]
        for ai in a])
    return out.squeeze()

def sound_horizon(a, bg):
    """Comoving sound horizon r_s(a) = int_0^a c_s da'/(a'^2 H), in Mpc."""
    def integrand(ap):
        R = 0.75 * bg['grhob'] * ap / bg['grhog']
        return 1.0 / np.sqrt(total_density_a4(ap, bg) * (1.0 + R))
    return integrate.quad(integrand, 0, a)[0]


### Cell: `build_background`


In [ ]:
def build_background(params):
    """Top-level: prepare the background for downstream use."""
    bg = init_background(params)
    bg['tau0']   = float(conformal_time(1.0, bg))
    a_eq         = (bg['grhog'] + bg['grhornomass']) / (bg['grhoc'] + bg['grhob'])
    bg['a_eq']   = a_eq
    bg['z_eq']   = 1.0 / a_eq - 1.0
    bg['tau_eq'] = float(conformal_time(a_eq, bg))
    return bg


### Cell: run it


In [ ]:
bg = build_background(params)
print(f"H_0     = {bg['H0']:.4e} 1/Mpc = {bg['H0']*c_km_s:.2f} km/s/Mpc")
print(f"tau_0   = {bg['tau0']:.1f} Mpc")
print(f"z_eq    = {bg['z_eq']:.0f}")
print(f"tau_eq  = {bg['tau_eq']:.2f} Mpc")


### Cell: plot $H(z)/H_0$


In [ ]:
a_grid = np.geomspace(1e-7, 1.0, 400)
z_grid = 1.0 / a_grid - 1.0
H_grid = np.array([hubble(ai, bg) for ai in a_grid])

# Where does matter equal Lambda?
rho_m = (bg['grhoc'] + bg['grhob']) / a_grid**3
rho_L = bg['grhov'] * np.ones_like(a_grid)
a_lambda = a_grid[np.argmin(np.abs(rho_m - rho_L))]
z_lambda = 1.0 / a_lambda - 1.0

fig, ax = plt.subplots()
sel = (z_grid > 1e-3) & (z_grid < 1e5)
ax.loglog(z_grid[sel], H_grid[sel] / bg['H0'], 'k-', lw=1.6)
ax.axvline(bg['z_eq'], color='C3', ls=':', alpha=0.7)
ax.text(bg['z_eq']*1.3, 2, f"$z_\\mathrm{{eq}} = {bg['z_eq']:.0f}$", color='C3')
ax.axvline(z_lambda, color='C2', ls=':', alpha=0.7)
ax.text(z_lambda*1.3, 2, f"$z_\\Lambda = {z_lambda:.2f}$", color='C2')
ax.set_xlabel('Redshift $z$')
ax.set_ylabel('$H(z) / H_0$')
ax.set_title('Hubble expansion history')
ax.grid(True, which='both', alpha=0.3)
plt.show()


### Cell: plot $Omega_i(a)$


In [ ]:
rho_r   = (bg['grhog'] + bg['grhornomass']) / a_grid**4
rho_m   = (bg['grhoc'] + bg['grhob']) / a_grid**3
rho_L   = bg['grhov'] * np.ones_like(a_grid)
rho_tot = rho_r + rho_m + rho_L

fig, ax = plt.subplots()
ax.loglog(a_grid, rho_m/rho_tot, label='Matter',    color='C0', lw=1.6)
ax.loglog(a_grid, rho_r/rho_tot, label='Radiation', color='C3', lw=1.6)
ax.loglog(a_grid, rho_L/rho_tot, label=r'$\Lambda$', color='C2', lw=1.6)
ax.axvline(bg['a_eq'], color='gray', ls=':', alpha=0.5)
ax.axvline(a_lambda,   color='gray', ls=':', alpha=0.5)
ax.text(bg['a_eq']*1.3, 1.2e-5, r'$a_\mathrm{eq}$', color='gray')
ax.text(a_lambda*1.3,   1.2e-5, r'$a_\Lambda$',     color='gray')
ax.set_xlabel('Scale factor $a$')
ax.set_ylabel(r'$\Omega_i(a)$')
ax.set_title('Composition of the universe')
ax.set_ylim(1e-6, 2); ax.set_xlim(1e-7, 1)
ax.legend(loc='lower right'); ax.grid(True, which='both', alpha=0.3)
plt.show()


# Chapter 2 — Recombination


### Cell: atomic constants


In [ ]:
c_SI = c_km_s * 1e3                              # speed of light (m/s)

# Atomic transition wavenumbers (1/m)
L_H_ion   = 1.096787737e7                        # H ionization
L_H_alpha = 8.225916453e6                        # H Lyman-alpha
L_He1_ion = 1.98310772e7                         # HeI ionization
L_He2_ion = 4.389088863e7                        # HeII ionization
L_He_2s   = 1.66277434e7                         # HeI 2^1 S_0
L_He_2p   = 1.71134891e7                         # HeI 2^1 P_1

# Decay/transition rates
Lambda_2s1s = 8.2245809                          # H 2s->1s two-photon (s^-1)
Lambda_He   = 51.3                               # HeI 2s->1s two-photon (s^-1)
A2P_s       = 1.798287e9                         # HeI singlet Einstein A (s^-1)
A2P_t       = 177.58                             # HeI triplet Einstein A (s^-1)
L_He_2Pt    = 1.690871466e7                      # HeI triplet 2^3 P (1/m)
L_He_2St    = 1.5985597526e7                     # HeI triplet 2^3 S (1/m)
L_He2St_ion = 3.8454693845e6                     # HeI 2^3 S ionisation (1/m)
sigma_He_2Ps = 1.436289e-22                      # HeI singlet photoionisation sigma (m^2)
sigma_He_2Pt = 1.484872e-22                      # HeI triplet photoionisation sigma (m^2)

# RECFAST fudge factors (calibrated against full level codes)
RECFAST_fudge = 1.125
AGauss1, AGauss2 = -0.14, 0.079
zGauss1, zGauss2 = 7.28, 6.73
wGauss1, wGauss2 = 0.18, 0.33

# Derived constants
CR        = 2 * np.pi * m_e * k_B / h_P**2       # Saha coefficient (1/m^2/K)
CB1       = h_P * c_SI * L_H_ion / k_B           # H ionisation / k_B (K)
CB1_He1   = h_P * c_SI * L_He1_ion / k_B
CB1_He2   = h_P * c_SI * L_He2_ion / k_B
CDB       = h_P * c_SI * (L_H_ion - L_H_alpha) / k_B
CDB_He    = h_P * c_SI * (L_He1_ion - L_He_2s) / k_B
CK        = (1.0 / L_H_alpha)**3 / (8 * np.pi)    # lambda_alpha^3 / (8 pi) (m^3)
CK_He     = (1.0 / L_He_2p)**3 / (8 * np.pi)
CL        = h_P * c_SI * L_H_alpha / k_B
CL_He     = h_P * c_SI * L_He_2s / k_B
Bfact     = h_P * c_SI * (L_He_2p - L_He_2s) / k_B
CL_PSt    = h_P * c_SI * (L_He_2Pt - L_He_2St) / k_B
CB1_He2St = h_P * c_SI * L_He2St_ion / k_B
CL_He_2St = h_P * c_SI * L_He_2St / k_B
a_rad     = 4 * sigma_SB / c_SI                   # radiation constant
CT        = (8/3) * (sigma_T / (m_e * c_SI)) * a_rad   # Compton cooling
barssc0   = k_B / (m_H * c_SI**2)                 # baryon sound speed prefactor (1/K)


### Cell: `solve_recombination` -- part 1 -- Saha and RHS


In [ ]:
def solve_recombination(bg, params):
    """Solve the ionisation history x_e(z) using the RECFAST equations.

    Three-variable ODE for hydrogen ionisation x_H, helium ionisation x_He,
    and matter temperature T_m. Saha equilibrium at high z, switching to
    ODEs as each species departs from equilibrium.
    """
    T_cmb = bg['T_cmb']
    f_He  = bg['f_He']

    # Present-day hydrogen number density (m^-3)
    H100_SI      = 100 * 1e3 / Mpc_in_m
    rho_crit_100 = 3 * H100_SI**2 / (8 * np.pi * G)
    Nnow         = (1 - bg['Y_He']) * (params['omega_b_h2'] * rho_crit_100) / m_H

    # Hubble in SI for use inside the ODE
    omega_m = (params['omega_b_h2'] + params['omega_c_h2']) / params['h']**2
    a_eq    = (bg['grhog'] + bg['grhornomass']) / (bg['grhoc'] + bg['grhob'])
    z_eq    = 1.0 / a_eq - 1.0
    H0_SI   = bg['H0'] * c_SI / Mpc_in_m
    def Hz_SI(z):
        return hubble(1.0 / (1 + z), bg) * c_SI / Mpc_in_m

    # --- Saha equations for helium ---
    def saha_He2(z):
        """He++ -> He+ Saha: returns total x_e per H atom."""
        T   = T_cmb * (1 + z)
        rhs = (CR * T_cmb / (1 + z))**1.5 * np.exp(-CB1_He2 / T) / Nnow
        return 0.5 * (np.sqrt((rhs - 1 - f_He)**2
                              + 4 * (1 + 2 * f_He) * rhs) - (rhs - 1 - f_He))

    def saha_He1(z):
        """He+ -> He0 Saha: returns x_He = n(He+) / n_He."""
        T   = T_cmb * (1 + z)
        rhs = 4.0 * (CR * T_cmb / (1 + z))**1.5 * np.exp(-CB1_He1 / T) / Nnow
        x0  = 0.5 * (np.sqrt((rhs - 1)**2 + 4 * (1 + f_He) * rhs) - (rhs - 1))
        return min((x0 - 1.0) / f_He, 1.0)

    # --- RECFAST 3-variable RHS: dy/dz for y = [x_H, x_He, T_mat] ---
    def recfast_rhs(z, y):
        x_H   = max(y[0], 0.0)
        x_He  = max(y[1], 0.0)
        T_mat = max(y[2], 0.5)
        x     = x_H + f_He * x_He
        T_rad = T_cmb * (1 + z)
        n_H   = Nnow * (1 + z)**3
        n_He  = f_He * n_H
        Hz    = Hz_SI(z)

        # ---- f1: Hydrogen Peebles equation ----
        t4    = T_mat / 1e4
        Rdown = 1e-19 * 4.309 * t4**(-0.6166) / (1 + 0.6703 * t4**0.5300)
        Rup   = Rdown * (CR * T_mat)**1.5 * np.exp(-CDB / T_mat)
        K     = CK / Hz * (1.0
                + AGauss1 * np.exp(-((np.log(1 + z) - zGauss1) / wGauss1)**2)
                + AGauss2 * np.exp(-((np.log(1 + z) - zGauss2) / wGauss2)**2))
        fu    = RECFAST_fudge
        n_1s  = n_H * max(1 - x_H, 1e-30)
        f1    = ((x * x_H * n_H * Rdown
                  - Rup * (1 - x_H) * np.exp(-CL / T_mat))
                 * (1 + K * Lambda_2s1s * n_1s)
                 / (Hz * (1 + z) * (1.0/fu + K * Lambda_2s1s * n_1s / fu
                                    + K * Rup * n_1s)))
        # ---- f2: Helium singlet ODE + triplet correction ----
        if x_He < 1e-15:
            f2 = 0.0
        else:
            T_0      = 10.0**0.477121
            T_1      = 10.0**5.114
            sq_0     = np.sqrt(T_mat / T_0)
            sq_1     = np.sqrt(T_mat / T_1)
            Rdown_He = 10.0**(-16.744) / (sq_0 * (1 + sq_0)**0.289
                                          * (1 + sq_1)**1.711)
            Rup_He   = 4.0 * Rdown_He * (CR * T_mat)**1.5 * np.exp(-CDB_He / T_mat)
            He_Boltz = np.exp(min(Bfact / T_mat, 500.0))

            n_He_ground = n_He * max(1 - x_He, 1e-30)
            tauHe_s     = A2P_s * CK_He * 3 * n_He_ground / Hz
            pHe_s       = ((1 - np.exp(-tauHe_s)) / tauHe_s
                           if tauHe_s > 1e-7 else 1.0 - tauHe_s / 2.0)

            if x_H < 0.9999999:
                Doppler_s = c_SI * L_He_2p * np.sqrt(2 * k_B * T_mat
                                / (m_H * not4 * c_SI**2))
                gamma_2Ps = (3 * A2P_s * f_He * (1 - x_He) * c_SI**2
                             / (np.sqrt(np.pi) * sigma_He_2Ps * 8 * np.pi
                                * Doppler_s * max(1 - x_H, 1e-30)
                                * (c_SI * L_He_2p)**2))
                AHcon_s = A2P_s / (1 + 0.36 * gamma_2Ps**0.86)
                K_He    = 1.0 / max((A2P_s * pHe_s + AHcon_s) * 3 * n_He_ground, 1e-300)
            else:
                K_He = 1.0 / max(A2P_s * pHe_s * 3 * n_He_ground, 1e-300)

            f2 = ((x * x_He * n_H * Rdown_He
                   - Rup_He * (1 - x_He) * np.exp(-CL_He / T_mat))
                  * (1 + K_He * Lambda_He * n_He_ground * He_Boltz)
                  / (Hz * (1 + z)
                     * (1 + K_He * (Lambda_He + Rup_He)
                        * n_He_ground * He_Boltz)))

            # Triplet channel
            if x_He > 5e-9:
                a_trip = 10.0**(-16.306); b_trip = 0.761
                Rdown_trip = a_trip / (sq_0 * (1 + sq_0)**(1.0 - b_trip)
                                       * (1 + sq_1)**(1.0 + b_trip))
                Rup_trip   = (4.0/3.0) * Rdown_trip * (CR * T_mat)**1.5 * np.exp(-CB1_He2St / T_mat)
                tauHe_t    = A2P_t * n_He_ground * 3 / (8 * np.pi * Hz * L_He_2Pt**3)
                pHe_t      = ((1 - np.exp(-tauHe_t)) / tauHe_t
                              if tauHe_t > 1e-7 else 1.0 - tauHe_t / 2.0)

                if x_H < 0.99999:
                    Doppler_t = c_SI * L_He_2Pt * np.sqrt(2 * k_B * T_mat
                                    / (m_H * not4 * c_SI**2))
                    gamma_2Pt = (3 * A2P_t * f_He * (1 - x_He) * c_SI**2
                                 / (np.sqrt(np.pi) * sigma_He_2Pt * 8 * np.pi
                                    * Doppler_t * max(1 - x_H, 1e-30)
                                    * (c_SI * L_He_2Pt)**2))
                    AHcon_t = A2P_t / (1 + 0.66 * gamma_2Pt**0.9) / 3.0
                    CfHe_t  = (A2P_t * pHe_t + AHcon_t) * np.exp(-CL_PSt / T_mat)
                else:
                    CfHe_t = A2P_t * pHe_t * np.exp(-CL_PSt / T_mat)
                denom = Rup_trip + CfHe_t
                CfHe_t = CfHe_t / denom if denom > 1e-300 else 0.0

                f2 += ((x * x_He * n_H * Rdown_trip
                        - (1 - x_He) * 3 * Rup_trip * np.exp(-CL_He_2St / T_mat))
                       * CfHe_t / (Hz * (1 + z)))

        # ---- f3: Matter temperature ----
        x_safe = max(x, 1e-30)
        timeTh = (1.0 / (CT * T_rad**4)) * (1 + x + f_He) / x_safe
        timeH  = 2.0 / (3.0 * H0_SI * (1 + z)**1.5)
        if timeTh < 1e-3 * timeH:
            # Tightly coupled: implicit form
            dHdz = (H0_SI**2 / (2 * Hz)) * omega_m * (
                4 * (1 + z)**3 / (1 + z_eq) + 3 * (1 + z)**2)
            epsilon = Hz * (1 + x + f_He) / (CT * T_rad**3 * x_safe)
            f3 = (T_cmb
                  + epsilon * (1 + f_He) / (1 + f_He + x)
                  * (f1 + f_He * f2) / x_safe
                  - epsilon * dHdz / Hz
                  + 3 * epsilon / (1 + z))
        else:
            # Loose coupling: Compton cooling + adiabatic
            f3 = (CT * T_rad**4 * x_safe / (1 + x + f_He)
                  * (T_mat - T_rad) / (Hz * (1 + z))
                  + 2 * T_mat / (1 + z))

        return [f1, f2, f3]
    # --- Build z grid ---
    z_start, z_end, nz = 10000, 0, 20000
    z_arr   = np.linspace(z_start, z_end, nz + 1)
    xH_arr  = np.ones(nz + 1)
    xHe_arr = np.ones(nz + 1)
    xe_total = np.empty(nz + 1)

    # --- Phase 1: Saha equilibrium ---
    he_ode_idx = None
    for i, z in enumerate(z_arr):
        if z > 8000.0:
            xH_arr[i] = 1.0; xHe_arr[i] = 1.0
            xe_total[i] = 1.0 + 2.0 * f_He
        elif z > 5000.0:
            xH_arr[i] = 1.0; xHe_arr[i] = 1.0
            xe_total[i] = saha_He2(z)
        elif z > 3500.0:
            xH_arr[i] = 1.0; xHe_arr[i] = 1.0
            xe_total[i] = 1.0 + f_He
        elif z > 0:
            x_He = saha_He1(z)
            xHe_arr[i] = x_He
            xH_arr[i]  = 1.0
            xe_total[i] = 1.0 + f_He * x_He
            if x_He < 0.99:
                he_ode_idx = i
                break
        else:
            break
    if he_ode_idx is None:
        he_ode_idx = len(z_arr) - 1

    # --- Phase 2: 3-variable ODE from He departure to z = 0 ---
    z_ode = z_arr[he_ode_idx:]
    z_ode = z_ode[z_ode >= 0]
    Tmat_arr = T_cmb * (1.0 + z_arr)
    if len(z_ode) > 1:
        y0 = [xH_arr[he_ode_idx], xHe_arr[he_ode_idx],
              T_cmb * (1 + z_ode[0])]
        sol = integrate.solve_ivp(
            recfast_rhs,
            [z_ode[0], z_ode[-1]], y0,
            t_eval=z_ode, method='Radau', rtol=1e-6, atol=1e-10,
            max_step=5.0,
        )
        n_sol = min(sol.y.shape[1], len(z_arr) - he_ode_idx)
        idx   = he_ode_idx + n_sol
        xH_arr[he_ode_idx:idx]   = sol.y[0, :n_sol]
        xHe_arr[he_ode_idx:idx]  = sol.y[1, :n_sol]
        Tmat_arr[he_ode_idx:idx] = sol.y[2, :n_sol]
    xe_total[he_ode_idx:] = xH_arr[he_ode_idx:] + f_He * xHe_arr[he_ode_idx:]
    return z_arr, xe_total, Tmat_arr


### Cell: `build_thermodynamics`


In [ ]:
def build_thermodynamics(bg, params):
    """Build thermodynamic tables: opacity, optical depth, visibility function,
    plus derived recombination quantities (z_*, tau_*, r_s, k_D, ...)."""
    z_arr, xe_arr, Tmat_rec = solve_recombination(bg, params)

    # Reionisation layer: CAMB tanh model
    f_He        = bg['f_He']
    delta_z     = 0.5
    he_reion_z  = 3.5
    he_reion_dz = 0.4
    he_reion_zstart = he_reion_z + 5 * he_reion_dz
    f_re        = 1.0 + f_He
    z_rev, xe_rev = z_arr[::-1], xe_arr[::-1]

    def build_reion_xe(z_eval, z_re):
        z_reion_start = z_re + 8 * delta_z
        x_e_freeze    = np.interp(z_reion_start, z_rev, xe_rev)
        window_var_mid   = (1 + z_re)**1.5
        window_var_delta = 1.5 * (1 + z_re)**0.5 * delta_z
        xod = np.clip((window_var_mid - (1 + z_eval)**1.5) / window_var_delta, -100, 100)
        x_H_reion = (f_re - x_e_freeze) * (np.tanh(xod) + 1) / 2 + x_e_freeze
        xod_he    = np.clip((he_reion_z - z_eval) / he_reion_dz, -100, 100)
        x_he_extra = np.where(z_eval < he_reion_zstart,
                              f_He * (np.tanh(xod_he) + 1) / 2, 0.0)
        return x_H_reion + x_he_extra

    def compute_reion_optical_depth(z_re):
        z_reion_start = z_re + 8 * delta_z
        def integrand(z):
            a = 1.0 / (1 + z)
            return build_reion_xe(z, z_re) * bg['akthom'] * dtau_da(a, bg)
        return integrate.quad(integrand, 0, z_reion_start, limit=200)[0]

    # Solve for z_re matching the target reionisation tau
    def tau_residual(z_re):
        return compute_reion_optical_depth(z_re) - params['tau_reion']
    z_re = optimize.brentq(tau_residual, 2.0, 30.0, xtol=1e-8, rtol=1e-8)

    # Refine the z grid around z_re
    z_reion_lo = max(0.01, z_re - 8 * delta_z)
    z_reion_hi = z_re + 8 * delta_z
    z_dense    = np.linspace(z_reion_hi, z_reion_lo, 200)
    mask       = (z_arr > z_reion_hi) | (z_arr < z_reion_lo)
    z_new      = np.sort(np.concatenate([z_arr[mask], z_dense]))[::-1]
    xe_new     = np.interp(z_new[::-1], z_arr[::-1], xe_arr[::-1])[::-1]
    Tmat_new   = np.interp(z_new[::-1], z_arr[::-1], Tmat_rec[::-1])[::-1]
    z_arr, xe_arr, Tmat_rec = z_new, xe_new, Tmat_new

    # Final x_e: max of recombination + reionisation
    xe_final = np.maximum(build_reion_xe(z_arr, z_re), xe_arr)

    # Conformal time grid (monotonically increasing in tau)
    a_arr   = 1.0 / (1 + z_arr)
    tau_arr = conformal_time(a_arr, bg)

    # Opacity, optical depth, visibility
    opacity     = xe_final * bg['akthom'] / a_arr**2
    tau_optical = -np.flip(integrate.cumulative_trapezoid(
        np.flip(opacity), np.flip(tau_arr), initial=0))
    exptau     = np.exp(-tau_optical)
    visibility = opacity * exptau

    # Assemble the thermo dictionary
    thermo = {
        'z_arr': z_arr, 'a_arr': a_arr, 'tau_arr': tau_arr,
        'xe': xe_final, 'Tmat': Tmat_rec,
        'opacity': opacity, 'tau_optical': tau_optical,
        'exptau': exptau, 'visibility': visibility,
        'z_reion': z_re,
    }
    thermo['opacity_interp']    = interpolate.CubicSpline(tau_arr, opacity)
    thermo['exptau_interp']     = interpolate.CubicSpline(tau_arr, exptau)
    thermo['visibility_interp'] = interpolate.CubicSpline(tau_arr, visibility)

    # Baryon sound speed
    dlnT_dln_a = np.gradient(np.log(np.maximum(Tmat_rec, 1e-30)), np.log(a_arr))
    barssc = barssc0 * (1.0 - 0.75 * bg['Y_He'] + (1.0 - bg['Y_He']) * xe_arr)
    cs2_b  = np.maximum(barssc * Tmat_rec * (1.0 - dlnT_dln_a / 3.0), 0.0)
    thermo['cs2_b'] = cs2_b

    # Peak of g(tau): the surface of last scattering
    peak_idx = np.argmax(visibility)
    thermo['z_star']   = z_arr[peak_idx]
    thermo['tau_star'] = tau_arr[peak_idx]
    a_star             = 1.0 / (1.0 + thermo['z_star'])

    # Sound horizon r_s and Silk damping scale k_D
    thermo['r_s'] = sound_horizon(a_star, bg)
    a_grid       = np.linspace(a_arr[0], a_star, 5000)
    R            = 0.75 * bg['grhob'] * a_grid / bg['grhog']
    dtauda_grid  = dtau_da(a_grid, bg)
    kappa_dot    = np.maximum(np.interp(a_grid, a_arr, xe_final) * bg['akthom'] / a_grid**2, 1e-30)
    integrand_D  = (R**2 + 16.0*(1.0+R)/15.0) / (6.0*(1.0+R)**2 * kappa_dot) * dtauda_grid
    thermo['k_D'] = 1.0 / np.sqrt(np.trapz(integrand_D, a_grid))

    # FWHM widths of the visibility peaks (needed by make_k_grid / make_tau_grid)
    fwhm_to_sigma = 2.0 * np.sqrt(2.0 * np.log(2.0))
    half_max  = visibility[peak_idx] / 2.0
    tau_left  = np.interp(half_max, visibility[:peak_idx+1], tau_arr[:peak_idx+1])
    tau_right = np.interp(half_max, visibility[peak_idx:][::-1], tau_arr[peak_idx:][::-1])
    thermo['delta_tau_rec'] = (tau_right - tau_left) / fwhm_to_sigma

    z_rev, tau_rev = z_arr[::-1], tau_arr[::-1]
    thermo['tau_reion'] = np.interp(z_re, z_rev, tau_rev)
    thermo['delta_tau_reion'] = abs(
        np.interp(z_re + 6*delta_z, z_rev, tau_rev)
        - np.interp(max(0.01, z_re - 6*delta_z), z_rev, tau_rev)) / fwhm_to_sigma

    return thermo


### Cell: build the thermodynamics


In [ ]:
th = build_thermodynamics(bg, params)
print(f"z_*       = {th['z_star']:.0f}")
print(f"tau_*     = {th['tau_star']:.1f} Mpc")
print(f"r_s(tau_*) = {th['r_s']:.1f} Mpc")
print(f"k_D       = {th['k_D']:.3f} Mpc^-1")
print(f"z_reion   = {th['z_reion']:.2f}")


### Cell: plot $x_e(z)$


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
sel = (th['z_arr'] > 1) & (th['z_arr'] < 1e4)
ax.loglog(th['z_arr'][sel], th['xe'][sel], 'k-', lw=1.6)
ax.invert_xaxis()
ax.set_xlim(1e4, 1)
ax.set_xlabel(r'Redshift $z$')
ax.set_ylabel(r'Free electron fraction $x_e$')
ax.set_title('Recombination + reionisation history')
ax.grid(True, which='both', alpha=0.3)
plt.show()


### Cell: plot $g(tau)$ and $dotkappa(tau)$


In [ ]:
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(7, 6))
sel = th['tau_arr'] > 0
axes[0].semilogx(th['tau_arr'][sel], th['visibility'][sel], 'k-', lw=1.4)
axes[0].set_ylabel(r'Visibility $g(\tau)$')
axes[0].set_title('Visibility function and Thomson opacity')
axes[0].grid(True, which='both', alpha=0.3)
axes[1].loglog(th['tau_arr'][sel], th['opacity'][sel], 'k-', lw=1.4)
axes[1].set_ylabel(r'$\dot\kappa\, [\mathrm{Mpc}^{-1}]$')
axes[1].set_xlabel(r'Conformal time $\tau\, [\mathrm{Mpc}]$')
axes[1].grid(True, which='both', alpha=0.3)
plt.show()


# Chapter 3 — Perturbations


### Cell: state-vector layout and hierarchy parameters


In [ ]:
# Hierarchy truncation (increase for higher ell_max accuracy)
LMAXG   = 15      # photon temperature: Theta_0 ... Theta_LMAXG
LMAXPOL = 15      # photon polarisation: E_2 ... E_LMAXPOL
LMAXNR  = 15      # massless neutrinos: N_0 ... N_LMAXNR

# Indices into the flat state vector
IX_ETAK = 0
IX_CLXC = 1
IX_CLXB = 2
IX_VB   = 3
IX_G    = 4                                   # Theta_0 at IX_G, Theta_1 at IX_G+1, ...
IX_POL  = IX_G + LMAXG + 1                    # E_2 at IX_POL, E_3 at IX_POL+1, ...
IX_R    = IX_POL + LMAXPOL - 1                # N_0 at IX_R, N_1 at IX_R+1, ...
NVAR    = IX_R + LMAXNR + 1


### Cell: cubic-spline evaluator


In [ ]:
@_jit
def _cubic_eval(x_knots, coeffs, t):
    """Evaluate a scipy.interpolate.CubicSpline at point t."""
    n = x_knots.shape[0] - 1
    lo, hi = 0, n - 1
    while lo < hi:
        mid = (lo + hi) >> 1
        if x_knots[mid + 1] < t:
            lo = mid + 1
        else:
            hi = mid
    dt = t - x_knots[lo]
    return ((coeffs[0, lo] * dt + coeffs[1, lo]) * dt
            + coeffs[2, lo]) * dt + coeffs[3, lo]


### Cell: `init_perturbation_grid`


In [ ]:
def init_perturbation_grid(bg, thermo):
    """Precompute a(tau) and background quantities on a fine conformal-time grid."""
    # Build tau(a) by cumulative integration of dtau/da
    a_grid       = np.logspace(-9, 0, 10000)
    dtauda_grid  = np.array([dtau_da(a, bg) for a in a_grid])
    tau_grid     = integrate.cumulative_trapezoid(dtauda_grid, a_grid, initial=0)
    a_of_tau     = interpolate.CubicSpline(tau_grid, a_grid)

    # Radiation-era expansion rate
    grho_rad = bg['grhog'] + bg['grhornomass']
    adotrad  = np.sqrt(grho_rad / 3.0)

    # Extend opacity, sound speed below the thermo grid (full ionisation)
    tau_thermo  = thermo['tau_arr']
    tau_early   = tau_grid[tau_grid < tau_thermo[0]]
    a_early     = a_of_tau(tau_early)
    tau_ext     = np.concatenate([tau_early, tau_thermo])

    opac_early  = (1.0 + bg['f_He']) * bg['akthom'] / a_early**2
    opacity_interp = interpolate.CubicSpline(
        tau_ext, np.concatenate([opac_early, thermo['opacity']]))

    xe_early     = 1.0 + 2.0 * bg['f_He']
    barssc_early = barssc0 * (1.0 - 0.75 * bg['Y_He']
                              + (1.0 - bg['Y_He']) * xe_early)
    cs2_early    = (4.0 / 3.0) * barssc_early * bg['T_cmb'] / a_early
    cs2_interp   = interpolate.CubicSpline(
        tau_ext, np.concatenate([cs2_early, thermo['cs2_b']]))

    return {
        'sp_a_x': a_of_tau.x,    'sp_a_c': a_of_tau.c,
        'sp_op_x': opacity_interp.x, 'sp_op_c': opacity_interp.c,
        'sp_cs_x': cs2_interp.x, 'sp_cs_c': cs2_interp.c,
        'bg_vec': np.array([bg['grhog'], bg['grhornomass'], bg['grhoc'],
                            bg['grhob'], bg['grhov']]),
        'adotrad': adotrad, 'grho_rad': grho_rad, 'tau0': bg['tau0'],
    }


### Cell: `adiabatic_ics`


In [ ]:
def adiabatic_ics(k, tau_start, bg, pgrid):
    """Adiabatic initial conditions on super-horizon scales (k*tau << 1)."""
    x  = k * tau_start
    x2 = x * x

    grho_rad = pgrid['grho_rad']
    Rv       = bg['grhornomass'] / grho_rad      # rho_nu/(rho_nu + rho_gamma)
    Rp15     = 4 * Rv + 15

    om    = (bg['grhob'] + bg['grhoc']) / np.sqrt(3.0 * grho_rad)
    omtau = om * tau_start

    y0 = np.zeros(NVAR)

    # Metric variable etak = k * eta
    y0[IX_ETAK] = -k * (1.0 - x2 / 12.0 * (-10.0 / Rp15 + 1.0))

    # Photon monopole and dipole
    clxg_init = x2 / 3.0 * (1.0 - omtau / 5.0)
    qg_init   = x2 * x / 27.0 * (1.0 - omtau / 5.0)
    y0[IX_G]     = clxg_init
    y0[IX_G + 1] = qg_init

    # CDM and baryon density: delta = 3/4 delta_gamma for adiabatic mode
    y0[IX_CLXC] = 0.75 * clxg_init
    y0[IX_CLXB] = 0.75 * clxg_init
    y0[IX_VB]   = 0.75 * qg_init

    # Massless neutrinos
    y0[IX_R]     = clxg_init
    y0[IX_R + 1] = (4 * Rv + 23) / Rp15 * x2 * x / 27.0
    y0[IX_R + 2] = -4.0 / 3.0 * x2 / Rp15 * (1.0 + omtau / 4.0
                                              * (4*Rv - 5) / (2*Rv + 15))
    if LMAXNR >= 3:
        y0[IX_R + 3] = -4.0 / 21.0 / Rp15 * x2 * x

    # All higher multipoles and polarisation start at zero
    return y0


### Cell: common background andEinstein terms (helper)


In [ ]:
@_jit
def _common_terms(tau, y, k, bg_vec, sp_a_x, sp_a_c):
    """Background and Einstein-source terms used by both RHS and source builders."""
    grhog, grhornomass, grhoc, grhob, grhov = bg_vec
    a = _cubic_eval(sp_a_x, sp_a_c, tau)
    a2 = a * a
    grhog_t = grhog / a2
    grhor_t = grhornomass / a2
    grhoc_t = grhoc / a
    grhob_t = grhob / a
    grho_a2 = grhog_t + grhor_t + grhoc_t + grhob_t + grhov * a2
    adotoa  = np.sqrt(grho_a2 / 3.0)

    etak = y[IX_ETAK]
    clxc = y[IX_CLXC]; clxb = y[IX_CLXB]; vb = y[IX_VB]
    clxg = y[IX_G];    qg = y[IX_G + 1];   pig = y[IX_G + 2]
    clxr = y[IX_R];    qr = y[IX_R + 1];   pir = y[IX_R + 2]

    k2    = k * k
    dgrho = grhob_t*clxb + grhoc_t*clxc + grhog_t*clxg + grhor_t*clxr
    dgq   = grhob_t*vb + grhog_t*qg + grhor_t*qr
    z     = (0.5 * dgrho / k + etak) / adotoa
    sigma = z + 1.5 * dgq / k2

    return (a, adotoa, grhog_t, grhor_t, grhoc_t, grhob_t,
            dgrho, dgq, z, sigma,
            etak, clxc, clxb, vb, clxg, qg, pig, clxr, qr, pir)


### Cell: `boltzmann_derivs` -- the Boltzmann RHS


In [ ]:
@_jit
def boltzmann_derivs(tau, y, k, bg_vec, sp_a_x, sp_a_c,
                     sp_op_x, sp_op_c, sp_cs_x, sp_cs_c):
    """dy/d(tau): the full Einstein-Boltzmann system in synchronous gauge."""
    (a, adotoa, grhog_t, grhor_t, grhoc_t, grhob_t,
     dgrho, dgq, z, sigma,
     etak, clxc, clxb, vb, clxg, qg, pig, clxr, qr, pir) = \
        _common_terms(tau, y, k, bg_vec, sp_a_x, sp_a_c)
    opacity = max(_cubic_eval(sp_op_x, sp_op_c, tau), 1e-30)
    cs2_b   = max(_cubic_eval(sp_cs_x, sp_cs_c, tau), 0.0)
    photbar = grhog_t / grhob_t
    pb43    = 4.0 / 3.0 * photbar
    delta_p_b = cs2_b * clxb

    tight_coupling = (k / opacity < 0.01) and (1.0 / (opacity * tau) < 0.01)
    E2      = y[IX_POL] if LMAXPOL >= 2 else 0.0
    cothxor = 1.0 / tau

    dy = np.zeros(NVAR)

    # --- Metric: dot{eta} k = (1/2) dgq ---
    dy[IX_ETAK] = 0.5 * dgq
    # --- CDM (at rest in this gauge) ---
    dy[IX_CLXC] = -k * z
    # --- Baryons: continuity ---
    dy[IX_CLXB] = -k * (z + vb)

    if tight_coupling:
        # Tight coupling: photon-baryon fluid locked together
        pig_tc = 32.0 / 45.0 * k / opacity * (sigma + vb)
        polter = pig_tc / 4.0

        vbdot  = (-adotoa * vb + k * delta_p_b
                   + k / 4.0 * pb43 * (clxg - 2.0 * pig_tc)) / (1.0 + pb43)
        dy[IX_VB] = vbdot

        dy[IX_G] = -k * (4.0 / 3.0 * z + qg)
        qgdot = 4.0 / 3.0 * (-vbdot - adotoa * vb
                              + k * delta_p_b) / pb43 + k / 3.0 * clxg \
                              - 2.0 * k / 3.0 * pig_tc
        dy[IX_G + 1] = qgdot
        dy[IX_G + 2] = opacity * (pig_tc - pig)

        if LMAXPOL >= 2:
            dy[IX_POL] = opacity * (pig_tc / 4.0 - E2)

    else:
        # Full Boltzmann hierarchy
        polter = pig / 10.0 + 9.0 / 15.0 * E2
        vbdot  = -adotoa * vb + k * delta_p_b \
                  - photbar * opacity * (4.0 / 3.0 * vb - qg)
        dy[IX_VB] = vbdot

        dy[IX_G] = -k * (4.0 / 3.0 * z + qg)
        qgdot = 4.0 / 3.0 * (-vbdot - adotoa * vb + k * delta_p_b) / pb43 \
                  + k / 3.0 * clxg - 2.0 * k / 3.0 * pig
        dy[IX_G + 1] = qgdot

        Theta3 = y[IX_G + 3] if LMAXG >= 3 else 0.0
        dy[IX_G + 2] = (2.0 * k / 5.0 * qg - 3.0 * k / 5.0 * Theta3
                        - opacity * (pig - polter) + 8.0 / 15.0 * k * sigma)

        for l in range(3, LMAXG):
            dy[IX_G + l] = (k * l / (2*l + 1) * y[IX_G + l - 1]
                            - k * (l + 1) / (2*l + 1) * y[IX_G + l + 1]
                            - opacity * y[IX_G + l])
        # Free-streaming closure
        dy[IX_G + LMAXG] = (k * y[IX_G + LMAXG - 1]
                             - (LMAXG + 1) * cothxor * y[IX_G + LMAXG]
                             - opacity * y[IX_G + LMAXG])

        # E-mode polarisation
        E3 = y[IX_POL + 1] if LMAXPOL >= 3 else 0.0
        dy[IX_POL] = -opacity * (E2 - polter) - k / 3.0 * E3

        for l in range(3, LMAXPOL):
            idx = IX_POL + l - 2
            polfac_l = (l + 3) * (l - 1) / (l + 1)
            dy[idx] = (-opacity * y[idx]
                       + k * l / (2*l + 1) * y[idx - 1]
                       - polfac_l * k / (2*l + 1) * y[idx + 1])

        idx_last = IX_POL + LMAXPOL - 2
        dy[idx_last] = (-opacity * y[idx_last]
                        + k * LMAXPOL / (2*LMAXPOL + 1) * y[idx_last - 1]
                        - (LMAXPOL + 3) * cothxor * y[idx_last])

    # --- Massless neutrinos ---
    dy[IX_R]     = -k * (4.0 / 3.0 * z + qr)
    dy[IX_R + 1] = k / 3.0 * (clxr - 2.0 * pir)
    N3 = y[IX_R + 3] if LMAXNR >= 3 else 0.0
    dy[IX_R + 2] = 2.0 * k / 5.0 * qr - 3.0 * k / 5.0 * N3 + 8.0 / 15.0 * k * sigma

    for l in range(3, LMAXNR):
        dy[IX_R + l] = (k * l / (2*l + 1) * y[IX_R + l - 1]
                        - k * (l + 1) / (2*l + 1) * y[IX_R + l + 1])
    dy[IX_R + LMAXNR] = (k * y[IX_R + LMAXNR - 1]
                          - (LMAXNR + 1) * cothxor * y[IX_R + LMAXNR])

    return dy


### Cell: `evolve_k`


In [ ]:
def evolve_k(k, bg, thermo, pgrid, tau_out):
    """Evolve perturbations for a single wavenumber k from tau_start to tau_out[-1].
    Returns sol.y of shape (NVAR, len(tau_out))."""
    tau_start = min(0.01 / k, tau_out[0] * 0.5)
    tau_start = max(tau_start, 0.1)
    y0 = adiabatic_ics(k, tau_start, bg, pgrid)

    bg_vec  = pgrid['bg_vec']
    sp_a_x  = pgrid['sp_a_x'];  sp_a_c  = pgrid['sp_a_c']
    sp_op_x = pgrid['sp_op_x']; sp_op_c = pgrid['sp_op_c']
    sp_cs_x = pgrid['sp_cs_x']; sp_cs_c = pgrid['sp_cs_c']

    sol = integrate.solve_ivp(
        lambda tau, y: boltzmann_derivs(
            tau, y, k, bg_vec, sp_a_x, sp_a_c, sp_op_x, sp_op_c, sp_cs_x, sp_cs_c),
        [tau_start, tau_out[-1]], y0, t_eval=tau_out,
        method='LSODA', rtol=1e-5, atol=1e-8, max_step=20.0)
    if not sol.success:
        raise RuntimeError(f"ODE solver failed for k={k:.4e}: {sol.message}")
    return sol.y


### Cell: evolve one mode


In [ ]:
pgrid   = init_perturbation_grid(bg, th)
k       = 0.05
tau_out = np.geomspace(0.5, bg['tau0'] * 0.999, 600)
Y       = evolve_k(k, bg, th, pgrid, tau_out)

# Extract individual variables
etak = Y[IX_ETAK]
clxc = Y[IX_CLXC]; clxb = Y[IX_CLXB]; vb = Y[IX_VB]
clxg = Y[IX_G];    qg   = Y[IX_G + 1]; pig = Y[IX_G + 2]

# Scale factor at each tau (for the x-axis)
a_of_tau = np.array([_cubic_eval(pgrid['sp_a_x'], pgrid['sp_a_c'], t)
                     for t in tau_out])


### Cell: plot the perturbations


In [ ]:
fig, axes = plt.subplots(3, 1, sharex=True, figsize=(7, 8))
m = (a_of_tau > 1e-6) & (a_of_tau < 1.0)

axes[0].semilogx(a_of_tau[m], clxg[m], label=r'$\delta_\gamma$', color='C0')
axes[0].semilogx(a_of_tau[m], clxc[m], label=r'$\delta_c$',     color='C1')
axes[0].semilogx(a_of_tau[m], clxb[m], label=r'$\delta_b$',     color='C2')
axes[0].set_ylabel('Density contrast')
axes[0].set_title(rf'Perturbation evolution at $k = {k}$ Mpc$^{{-1}}$')
axes[0].legend(loc='upper left'); axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(-3, 8)

axes[1].semilogx(a_of_tau[m], qg[m] * 3 / 4, label=r'$v_\gamma$', color='C0')
axes[1].semilogx(a_of_tau[m], vb[m],          label=r'$v_b$',      color='C2')
axes[1].set_ylabel('Velocities')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].semilogx(a_of_tau[m], pig[m], 'C0-', label=r'$\pi_\gamma$')
ax2 = axes[2].twinx()
ax2.semilogx(a_of_tau[m], etak[m] / k, 'C1-', label=r'$\eta$')
axes[2].set_ylabel(r'$\pi_\gamma$', color='C0')
ax2.set_ylabel(r'$\eta$ (metric)', color='C1')
axes[2].set_xlabel('Scale factor $a$')
axes[2].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


# Chapter 4 — Line of sight


### Cell: `build_sources`


In [ ]:
def build_sources(tau, y, k, pgrid, thermo):
    """Source function building blocks at a single (k, tau) point.

    Returns (ISW, monopole, sigma+vb, vis, polter, source_E) -- the
    raw pieces that get combined into the three temperature Bessel
    channels and the E-mode source.
    """
    (a, adotoa, grhog_t, grhor_t, grhoc_t, grhob_t,
     dgrho, dgq, z, sigma,
     etak, clxc, clxb, vb, clxg, qg, pig, clxr, qr, pir) = _common_terms(
        tau, y, k, pgrid['bg_vec'], pgrid['sp_a_x'], pgrid['sp_a_c'])
    opacity = max(float(_cubic_eval(pgrid['sp_op_x'], pgrid['sp_op_c'], tau)),
                  1e-30)
    k2 = k * k

    E2 = y[IX_POL] if LMAXPOL >= 2 else 0.0

    # --- Weyl potential phi = (Phi + Psi)/2 in synchronous gauge ---
    dgpi = grhog_t * pig + grhor_t * pir
    phi  = -((dgrho + 3.0 * dgq * adotoa / k) + dgpi) / (2.0 * k2)

    # --- Compute phi_dot directly from hierarchy equations ---
    polter = pig / 10.0 + 9.0 / 15.0 * E2
    Theta3 = y[IX_G + 3] if LMAXG >= 3 else 0.0
    N3     = y[IX_R + 3] if LMAXNR >= 3 else 0.0
    pigdot = (2*k/5*qg - 3*k/5*Theta3 - opacity*(pig - polter) + 8*k*sigma/15)
    pirdot = (2*k/5*qr - 3*k/5*N3 + 8*k*sigma/15)
    pidot_sum  = grhog_t * pigdot + grhor_t * pirdot
    diff_rhopi = pidot_sum - 4.0 * adotoa * dgpi
    gpres_plus_grho = (4.0/3.0) * (grhog_t + grhor_t) + grhoc_t + grhob_t
    phidot = 0.5 * (adotoa * (-dgpi - 2.0 * k2 * phi) + dgq * k
                     - diff_rhopi + k * sigma * gpres_plus_grho) / k2

    # --- Visibility and exp(-tau) at this time ---
    tau_min, tau_max = thermo['tau_arr'][0], thermo['tau_arr'][-1]
    if tau < tau_min or tau > tau_max:
        vis    = 0.0
        exptau = np.exp(-thermo['tau_optical'][0]) if tau < tau_min else 1.0
    else:
        vis    = float(thermo['visibility_interp'](tau))
        exptau = float(thermo['exptau_interp'](tau))

    # --- Three source components ---
    ISW       = 2.0 * phidot * exptau                       # 2 phi_dot e^{-kappa}
    monopole  = -etak / k + 2.0 * phi + clxg / 4.0          # Theta_0 + Psi_W + (3/8) delta_b...
    chi       = pgrid['tau0'] - tau
    source_E  = 15.0 / 8.0 * vis * polter / (chi**2 * k2) if chi > 0 else 0.0

    return ISW, monopole, sigma + vb, vis, polter, source_E


### Cell: `evolve_k` (source-function version)


In [ ]:
def evolve_k(k, bg, thermo, pgrid, tau_out):
    """Evolve perturbations for wavenumber k, return source functions on tau_out.
    Returns (src_j0, src_j1, src_j2, src_E)."""
    tau_start = min(0.01 / k, tau_out[0] * 0.5)
    tau_start = max(tau_start, 0.1)
    y0 = adiabatic_ics(k, tau_start, bg, pgrid)

    bg_vec  = pgrid['bg_vec']
    sp_a_x  = pgrid['sp_a_x'];  sp_a_c  = pgrid['sp_a_c']
    sp_op_x = pgrid['sp_op_x']; sp_op_c = pgrid['sp_op_c']
    sp_cs_x = pgrid['sp_cs_x']; sp_cs_c = pgrid['sp_cs_c']

    sol = integrate.solve_ivp(
        lambda tau, y: boltzmann_derivs(
            tau, y, k, bg_vec, sp_a_x, sp_a_c, sp_op_x, sp_op_c, sp_cs_x, sp_cs_c),
        [tau_start, tau_out[-1]], y0, t_eval=tau_out,
        method='LSODA', rtol=1e-5, atol=1e-8, max_step=20.0)
    if not sol.success:
        raise RuntimeError(f"ODE solver failed for k={k:.4e}: {sol.message}")

    ntau = len(tau_out)
    ISW_arr      = np.zeros(ntau)
    monopole_arr = np.zeros(ntau)
    sigma_vb_arr = np.zeros(ntau)
    vis_arr      = np.zeros(ntau)
    polter_arr   = np.zeros(ntau)
    src_E        = np.zeros(ntau)
    for i, tau in enumerate(tau_out):
        (ISW_arr[i], monopole_arr[i], sigma_vb_arr[i],
         vis_arr[i], polter_arr[i], src_E[i]) = build_sources(
            tau, sol.y[:, i], k, pgrid, thermo)

    # Assemble the three temperature channels
    src_j0 = ISW_arr + vis_arr * (monopole_arr + 0.625 * polter_arr)
    src_j1 = vis_arr * sigma_vb_arr
    src_j2 = 1.875 * vis_arr * polter_arr
    return src_j0, src_j1, src_j2, src_E


# Chapter 5 — Power spectra


### Cell: `make_k_grid`


In [ ]:
def make_k_grid(N, mode, bg, thermo, params,
                k_min=1e-5, k_max=0.5,
                ell_min=2, ell_max=2500, n_ell_samples=30,
                n_eval=5000):
    """Optimal non-uniform k-grid.
    mode='cl'  : optimised for the C_ell integral
    mode='ode' : optimised for source-function interpolation.
    """
    x = np.linspace(np.log(k_min), np.log(k_max), n_eval)
    k = np.exp(x)
    primordial    = k ** (params['n_s'] + 2)
    acoustic_curv = (1.0 / thermo['r_s']) ** 2

    if mode == "cl":
        damped   = primordial * np.exp(-1.0 * (k / thermo['k_D']) ** 2)
        sigma_k  = 1.0 / thermo['delta_tau_rec']
        chi_star = bg['tau0'] - thermo['tau_star']
        ells     = np.unique(np.geomspace(ell_min, ell_max,
                                           n_ell_samples).astype(int))
        raw_weight = np.zeros_like(k)
        for ell in ells:
            envelope = np.exp(-0.5 * ((k - ell / chi_star) / (3.0 * sigma_k)) ** 2)
            curv     = np.maximum(acoustic_curv, sigma_k ** 2) * envelope
            raw_weight += curv * damped
        floor = 1e-6 * np.max(raw_weight)
    else:
        damped      = primordial * np.exp(-1.0 * (k / thermo['k_D']) ** 2)
        smooth_curv = (k * thermo['r_s']) ** 2 / bg['tau_eq'] ** 2
        raw_weight  = np.maximum(acoustic_curv, smooth_curv) * damped
        floor       = 0.005 * np.max(raw_weight)

    density = (raw_weight + floor) ** (1.0 / 3.0)
    dx      = x[1] - x[0]
    cdf     = np.cumsum(density) * dx
    cdf    -= cdf[0]; cdf /= cdf[-1]
    grid    = np.exp(np.interp(np.linspace(0, 1, N), cdf, x))
    grid[0] = k_min; grid[-1] = k_max
    return grid


### Cell: `make_tau_grid`


In [ ]:
def make_tau_grid(N, k_max, bg, thermo,
                  tau_min=1.0, tau_max=None, n_eval=10000):
    """Optimal non-uniform tau-grid for the line-of-sight integral."""
    if tau_max is None:
        tau_max = bg['tau0']
    tau           = np.linspace(tau_min, tau_max, n_eval)
    tau_star      = thermo['tau_star']
    delta_tau_rec = thermo['delta_tau_rec']

    g_rec   = np.exp(-0.5 * ((tau - tau_star) / delta_tau_rec) ** 2)
    g_broad = np.exp(-0.5 * ((tau - tau_star) / thermo['r_s']) ** 2)
    weight  = g_rec / delta_tau_rec**2 + g_broad * (k_max / np.sqrt(3.0))**2
    g_reion = np.exp(-0.5 * ((tau - thermo['tau_reion'])
                              / thermo['delta_tau_reion']) ** 2)
    weight += 0.3 * g_reion / thermo['delta_tau_reion'] ** 2

    density = (weight + 0.005 * np.max(weight)) ** (1.0 / 3.0)
    dtau    = tau[1] - tau[0]
    cdf     = np.cumsum(density) * dtau
    cdf    -= cdf[0]; cdf /= cdf[-1]
    grid    = np.interp(np.linspace(0, 1, N), cdf, tau)
    grid[0] = tau_min; grid[-1] = tau_max
    return grid


### Cell: Bessel tables


In [ ]:
def _build_bessel_tables(ells_compute, x_max, dx):
    """Precompute j_l and j_{l+1} on a uniform x-grid."""
    n_x  = int(np.ceil(x_max / dx)) + 1
    x_lo = 0.0
    x_arr = x_lo + dx * np.arange(n_x)
    inv_dx = 1.0 / dx
    nell = len(ells_compute)
    jl_tab  = np.zeros((nell, n_x))
    jl1_tab = np.zeros((nell, n_x))
    for il, ell in enumerate(ells_compute):
        jl_tab[il]  = special.spherical_jn(ell,     x_arr)
        jl1_tab[il] = special.spherical_jn(ell + 1, x_arr)
    return x_lo, inv_dx, n_x, jl_tab, jl1_tab

@_jit
def _interp_uniform_table(x, x0, inv_dx, n_x, vals):
    """Linear interpolation on a uniform grid."""
    out = np.zeros_like(x)
    flat = x.ravel()
    flat_out = out.ravel()
    for i in range(flat.size):
        u = (flat[i] - x0) * inv_dx
        j = int(u)
        if 0 <= j < n_x - 1:
            t = u - j
            flat_out[i] = (1 - t) * vals[j] + t * vals[j + 1]
    return out


### Cell: `compute_spectra`


In [ ]:
def compute_spectra(bg, thermo, params, N_k_ode=200, N_k_fine=4000, N_tau=1000,
                    k_arr=None, k_fine=None, tau_out=None):
    """Main CMB pipeline: evolve all k modes, do LOS integration, assemble C_ell."""
    pgrid = init_perturbation_grid(bg, thermo)
    tau0  = bg['tau0']
    tau_star = thermo['tau_star']

    if k_arr is None:
        k_arr = make_k_grid(N_k_ode, "ode", bg, thermo, params)
    nk = len(k_arr)
    if k_fine is None:
        k_fine = make_k_grid(N_k_fine, "cl", bg, thermo, params,
                             k_min=k_arr[0], k_max=k_arr[-1])
    if tau_out is None:
        tau_out = make_tau_grid(N_tau, k_arr[-1], bg, thermo,
                                tau_min=1.0, tau_max=tau0 - 1)
    ntau = len(tau_out)

    # Warm up numba JIT (no-op without numba), then evolve all modes sequentially.
    # A parallel multiprocessing version is in nanocmb.py if you need it.
    boltzmann_derivs(tau_out[0], np.zeros(NVAR), k_arr[0],
                     pgrid['bg_vec'], pgrid['sp_a_x'], pgrid['sp_a_c'],
                     pgrid['sp_op_x'], pgrid['sp_op_c'],
                     pgrid['sp_cs_x'], pgrid['sp_cs_c'])
    results = [evolve_k(k, bg, thermo, pgrid, tau_out) for k in k_arr]

    sources_j0 = np.array([r[0] for r in results])
    sources_j1 = np.array([r[1] for r in results])
    sources_j2 = np.array([r[2] for r in results])
    sources_E  = np.array([r[3] for r in results])

    # Akima interpolation onto fine k-grid
    nk_fine = len(k_fine)
    lnk_ode  = np.log(k_arr)
    lnk_fine = np.log(k_fine)
    src_fine_j0 = np.zeros((nk_fine, ntau))
    src_fine_j1 = np.zeros((nk_fine, ntau))
    src_fine_j2 = np.zeros((nk_fine, ntau))
    src_fine_E  = np.zeros((nk_fine, ntau))
    for it in range(ntau):
        for src_in, src_out in [(sources_j0, src_fine_j0), (sources_j1, src_fine_j1),
                                 (sources_j2, src_fine_j2), (sources_E,  src_fine_E)]:
            src_out[:, it] = interpolate.Akima1DInterpolator(lnk_ode, src_in[:, it])(lnk_fine)

    # ell-grid: dense at low ell, sparser at high ell
    ell_max = params['ell_max']
    ells_compute = np.unique(np.concatenate([
        np.arange(2, 40, 2),
        np.arange(40, 200, 5),
        np.arange(200, ell_max + 1, 50),
    ]))
    ells_compute = ells_compute[ells_compute <= ell_max]
    nell = len(ells_compute)

    chi_arr  = tau0 - tau_out
    chi_star = tau0 - tau_star
    chi_max  = chi_arr.max()

    Delta_T  = np.zeros((nell, nk_fine))
    Delta_E  = np.zeros((nell, nk_fine))
    x_2d_full = k_fine[:, None] * chi_arr[None, :]

    x0_tab, inv_dx_tab, n_x_tab, jl_tab, jl1_tab = _build_bessel_tables(
        ells_compute, float(np.max(x_2d_full)) + 2.0, 0.03)

    def _compute_ell_transfer(il, ell):
        # Restrict the k range to where j_ell has support
        x_lo  = max(0.0, ell - 4.0 * ell**(1.0/3.0))
        k_lo  = x_lo / chi_max if chi_max > 0 else 0
        k_hi  = (ell + 2500) / chi_star if chi_star > 0 else k_fine[-1]
        ik_lo = max(0,       np.searchsorted(k_fine, k_lo) - 1)
        ik_hi = min(nk_fine, np.searchsorted(k_fine, k_hi) + 1)

        x_2d = x_2d_full[ik_lo:ik_hi, :]
        jl      = _interp_uniform_table(x_2d, x0_tab, inv_dx_tab, n_x_tab, jl_tab[il])
        jl_next = _interp_uniform_table(x_2d, x0_tab, inv_dx_tab, n_x_tab, jl1_tab[il])

        with np.errstate(divide='ignore', invalid='ignore'):
            inv_x   = np.where(x_2d > 1e-30, 1.0 / x_2d, 0.0)
            jl_d    = np.where(x_2d > 1e-30, ell * inv_x * jl - jl_next, 0.0)
            jl_dd   = np.where(x_2d > 1e-30,
                               -2.0 * inv_x * jl_d
                               + (ell * (ell + 1) * inv_x * inv_x - 1.0) * jl,
                               0.0)
        integrand_T = (src_fine_j0[ik_lo:ik_hi, :] * jl
                       + src_fine_j1[ik_lo:ik_hi, :] * jl_d
                       + src_fine_j2[ik_lo:ik_hi, :] * jl_dd)
        integrand_E = src_fine_E[ik_lo:ik_hi, :] * jl
        Delta_T[il, ik_lo:ik_hi] = np.trapz(integrand_T, tau_out, axis=1)
        Delta_E[il, ik_lo:ik_hi] = np.trapz(integrand_E, tau_out, axis=1)

    for il, ell in enumerate(ells_compute):
        _compute_ell_transfer(il, int(ell))

    # Primordial power and assembly: C_l = 4 pi int dlnk P(k) Delta_T^2
    Pk = params['A_s'] * (k_fine / params['k_pivot'])**(params['n_s'] - 1.0)
    Cl_TT, Cl_EE, Cl_TE = [np.trapz(Pk * d, lnk_fine, axis=1)
                            for d in (Delta_T**2, Delta_E**2, Delta_T * Delta_E)]

    # Normalise to D_l = l(l+1) C_l / (2 pi)  (E mode has extra spin-2 factor)
    ells_f = ells_compute.astype(float)
    norm    = 4.0 * np.pi * ells_f * (ells_f + 1) / (2.0 * np.pi)
    ctnorm  = (ells_f**2 - 1.0) * (ells_f + 2) * ells_f
    Cl_TT *= norm
    Cl_EE *= norm * ctnorm
    Cl_TE *= norm * np.sqrt(ctnorm)

    # Convert dimensionless (dT/T)^2 to mu K^2
    T0_muK2 = (params['T_cmb'] * 1e6) ** 2
    Cl_TT *= T0_muK2; Cl_EE *= T0_muK2; Cl_TE *= T0_muK2

    # Interpolate to all integer ell
    ells_all = np.arange(2, ell_max + 1)
    Dl_TT, Dl_EE, Dl_TE = [interpolate.CubicSpline(ells_compute, cl)(ells_all)
                            for cl in (Cl_TT, Cl_EE, Cl_TE)]

    return {
        'ells': ells_all, 'Dl_TT': Dl_TT, 'Dl_EE': Dl_EE, 'Dl_TE': Dl_TE,
        'k_fine': k_fine, 'ells_compute': ells_compute,
        'Delta_T': Delta_T, 'Delta_E': Delta_E,
    }


### Cell: compute the scalar spectra


In [ ]:
result = compute_spectra(bg, th, params)
ells   = result['ells']
Dl_TT  = result['Dl_TT']
Dl_EE  = result['Dl_EE']
Dl_TE  = result['Dl_TE']


### Cell: plot the scalar spectra


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ells, Dl_TT,         color='C0', label='TT')
ax.plot(ells, Dl_EE,         color='C1', label='EE')
ax.plot(ells, np.abs(Dl_TE), color='C2', label='|TE|')
ax.set_xlabel(r'Multipole $\ell$')
ax.set_ylabel(r'$\mathcal{D}_\ell\,[\mu K^2]$')
ax.set_title('Scalar CMB power spectra')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlim(2, ells[-1]); ax.legend(); ax.grid(True, which='both', alpha=0.3)
plt.show()


### Cell: compare with CAMB


In [ ]:
import camb
pars = camb.set_params(H0=100 * params['h'],
                       ombh2=params['omega_b_h2'],
                       omch2=params['omega_c_h2'],
                       As=params['A_s'], ns=params['n_s'],
                       tau=params['tau_reion'])
pars.set_for_lmax(params['ell_max'], lens_potential_accuracy=2)
camb_powers = camb.get_results(pars).get_cmb_power_spectra(
    pars, CMB_unit='muK', raw_cl=False)
unlensed = camb_powers['unlensed_scalar']  # shape (lmax+1, 4); cols = TT,EE,BB,TE

ells_camb = np.arange(unlensed.shape[0])
Dl_TT_camb = unlensed[:, 0]
Dl_EE_camb = unlensed[:, 1]
Dl_TE_camb = unlensed[:, 3]


### Cell: plot tutorial vs CAMB with residuals


In [ ]:
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(8, 6),
                         gridspec_kw={'height_ratios': [3, 1]})

# Top: spectra
axes[0].plot(ells,      Dl_TT,                  'k-',  lw=1.4, label='tutorial')
axes[0].plot(ells_camb, Dl_TT_camb,             'C3--', lw=1.0, label='CAMB')
axes[0].plot(ells,      Dl_EE,                  'k-',  lw=1.4)
axes[0].plot(ells_camb, Dl_EE_camb,             'C3--', lw=1.0)
axes[0].set_yscale('log'); axes[0].set_xscale('log')
axes[0].set_ylabel(r'$\mathcal{D}_\ell$ [$\mu K^2$]')
axes[0].set_xlim(2, params['ell_max'])
axes[0].legend(); axes[0].grid(True, which='both', alpha=0.3)
axes[0].set_title('Validation: tutorial vs CAMB')

# Residuals
common = np.intersect1d(ells, ells_camb)
ix_t   = np.isin(ells, common)
ix_c   = np.isin(ells_camb, common)
resid_TT = 100.0 * (Dl_TT[ix_t] - Dl_TT_camb[ix_c]) / Dl_TT_camb[ix_c]
resid_EE = 100.0 * (Dl_EE[ix_t] - Dl_EE_camb[ix_c]) / Dl_EE_camb[ix_c]
axes[1].plot(common, resid_TT, 'C0-', lw=1.0, label='TT')
axes[1].plot(common, resid_EE, 'C1-', lw=1.0, label='EE')
axes[1].axhline(0, color='gray', lw=0.5)
axes[1].set_xscale('log')
axes[1].set_xlabel(r'Multipole $\ell$')
axes[1].set_ylabel('residual [%]')
axes[1].set_ylim(-5, 5)
axes[1].legend(); axes[1].grid(True, which='both', alpha=0.3)
plt.show()


# Chapter 6 — Lensing


### Cell: `evolve_phi`


In [ ]:
def evolve_phi(k, bg, thermo, pgrid, tau_out):
    """Evolve mode k, return the Weyl potential Psi_W(tau) at each tau_out point."""
    tau_start = max(1e-4, min(1e-3 / k, tau_out[0] * 0.5))
    y0 = adiabatic_ics(k, tau_start, bg, pgrid)

    bg_vec  = pgrid['bg_vec']
    sp_a_x  = pgrid['sp_a_x'];  sp_a_c  = pgrid['sp_a_c']
    sp_op_x = pgrid['sp_op_x']; sp_op_c = pgrid['sp_op_c']
    sp_cs_x = pgrid['sp_cs_x']; sp_cs_c = pgrid['sp_cs_c']

    sol = integrate.solve_ivp(
        boltzmann_derivs, (tau_start, tau_out[-1]), y0,
        args=(k, bg_vec, sp_a_x, sp_a_c, sp_op_x, sp_op_c, sp_cs_x, sp_cs_c),
        t_eval=tau_out, method='LSODA', rtol=1e-6, atol=1e-10)
    if not sol.success:
        return np.zeros(len(tau_out))

    psi_arr = np.zeros(len(tau_out))
    for i, tau in enumerate(tau_out):
        y = sol.y[:, i]
        (a, adotoa, grhog_t, grhor_t, _, _,
         _, _, _, sigma,
         etak, _, _, _, _, _, pig, _, _, pir) = _common_terms(
            tau, y, k, bg_vec, sp_a_x, sp_a_c)
        dgpi = grhog_t * pig + grhor_t * pir
        eta  = etak / k
        # Psi_W = eta - (a'/a) sigma/k - 3 dgpi / (4 k^2)
        psi_arr[i] = eta - adotoa * sigma / k - 0.75 * dgpi / (k * k)
    return psi_arr


### Cell: `lensing_potential`


In [ ]:
def lensing_potential(params, bg, thermo, pgrid,
                      ells=None, N_k=40, N_chi=120, ell_los_max=5):
    """Compute C_L^{phi phi} via hybrid LOS (low L) + Limber (high L)."""
    tau_star = float(thermo['tau_star'])
    tau0     = float(bg['tau0'])
    chi_star = tau0 - tau_star

    if ells is None:
        ells = np.unique(np.concatenate([
            np.arange(2, 30),
            np.geomspace(30, 2500, 80).astype(int)]))
    ells = np.asarray(ells, dtype=int)

    ell_max      = int(ells.max())
    chi_min_eff  = 0.01 * chi_star
    k_max_needed = (ell_max + 0.5) / chi_min_eff
    k_max        = min(max(k_max_needed * 1.2, 1.0), 5.0)
    k_arr        = np.geomspace(1e-5, k_max, N_k)

    chi_arr    = np.linspace(0.001 * chi_star, 0.999 * chi_star, N_chi)
    tau_sorted = np.sort(tau0 - chi_arr)
    chi_sorted = tau0 - tau_sorted

    phi_grid_sorted = np.zeros((N_k, N_chi))
    for i_k, k in enumerate(k_arr):
        phi_grid_sorted[i_k, :] = evolve_phi(k, bg, thermo, pgrid, tau_sorted)
    phi_grid_asc = phi_grid_sorted[:, ::-1]

    A_s = params['A_s']; n_s = params['n_s']; k_pivot = params['k_pivot']

    # --- LOS branch (ell <= ell_los_max) ---
    ells_los = ells[ells <= ell_los_max]
    Cl_los   = np.zeros(len(ells_los))
    W_lens   = (chi_star - chi_sorted) / (chi_star * chi_sorted)
    for i_ell, ell in enumerate(ells_los):
        Delta = np.zeros(N_k)
        for i_k, k in enumerate(k_arr):
            jl = special.spherical_jn(int(ell), k * chi_sorted)
            integrand = -2.0 * phi_grid_sorted[i_k] * W_lens * jl
            Delta[i_k] = -np.trapz(integrand, chi_sorted)
        P_R = A_s * (k_arr / k_pivot)**(n_s - 1.0)
        Cl_los[i_ell] = 4.0 * np.pi * np.trapz(P_R * Delta**2, np.log(k_arr))

    # --- Limber branch (ell > ell_los_max) ---
    chi_asc    = chi_sorted[::-1]
    phi_interp = interpolate.RegularGridInterpolator(
        (np.log(k_arr), chi_asc), phi_grid_asc,
        bounds_error=False, fill_value=0.0)
    ells_limber = ells[ells > ell_los_max]
    Cl_limber   = np.zeros(len(ells_limber))
    for i_ell, ell in enumerate(ells_limber):
        nu     = ell + 0.5
        k_lim  = nu / chi_asc
        valid  = (k_lim >= k_arr[0]) & (k_lim <= k_arr[-1])
        if not np.any(valid):
            continue
        chi_v  = chi_asc[valid]
        k_v    = k_lim[valid]
        T_psi  = phi_interp((np.log(k_v), chi_v))
        Delta2_R = A_s * (k_v / k_pivot)**(n_s - 1.0)
        chiW2  = (chi_star - chi_v)**2 / (chi_star**2 * chi_v)
        Cl_limber[i_ell] = (8.0 * np.pi**2 / nu**3) * np.trapz(
            T_psi**2 * Delta2_R * chiW2, chi_v)

    Cl_phi = np.zeros(len(ells))
    Cl_phi[ells <= ell_los_max] = Cl_los
    Cl_phi[ells > ell_los_max]  = Cl_limber
    Dl_phi = (ells * (ells + 1.0))**2 * Cl_phi / (2.0 * np.pi)

    return {'ell': ells, 'Cl_phi_phi': Cl_phi, 'Dl_phi_phi': Dl_phi,
            'k_arr': k_arr, 'chi_star': chi_star}


### Cell: plot $C_L^{phiphi}$


In [ ]:
phi = lensing_potential(params, bg, th, init_perturbation_grid(bg, th))

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.loglog(phi['ell'], phi['Dl_phi_phi'], 'k-', lw=1.5)
ax.set_xlabel(r'$L$')
ax.set_ylabel(r'$[L(L+1)]^2 C_L^{\phi\phi}/(2\pi)$')
ax.set_title('CMB lensing potential power spectrum')
ax.set_xlim(2, 2500); ax.grid(True, which='both', alpha=0.3)
plt.show()


### Cell: factorials and the normalising prefactor


In [ ]:
from math import lgamma
from scipy.special import eval_legendre, eval_jacobi

def _log_fact(n):
    return lgamma(n + 1.0)

def _norm_factor(l, m, mp):
    """sqrt[(l-m)!(l+m)! / ((l-mp)!(l+mp)!)] computed via log-gamma."""
    log = 0.5 * (_log_fact(l - m) + _log_fact(l + m)
                 - _log_fact(l - mp) - _log_fact(l + mp))
    return np.exp(log)


### Cell: single-$ell$ Wigner evaluator (canonical s and s prime)


In [ ]:
def _d_general(l, m, mp, beta):
    """d^l_{m, mp}(beta) for arrays of beta. Assumes m >= |mp| and m + mp >= 0."""
    sin_h = np.sin(beta / 2.0)
    cos_h = np.cos(beta / 2.0)
    cos_b = np.cos(beta)
    norm  = _norm_factor(l, m, mp)
    s = sin_h ** (m - mp) if m != mp     else np.ones_like(beta)
    c = cos_h ** (m + mp) if (m + mp) != 0 else np.ones_like(beta)
    n = l - m
    if n < 0:
        return np.zeros_like(beta)
    jacobi = eval_jacobi(n, m - mp, m + mp, cos_b)
    return norm * s * c * jacobi


### Cell: `wigner_d_array`


In [ ]:
def wigner_d_array(l_max, m, mp, beta):
    """Compute d^l_{m, mp}(beta) for l = 0..l_max at all beta values.
    Returns array of shape (l_max + 1, len(beta))."""
    beta = np.atleast_1d(beta).astype(float)
    d = np.zeros((l_max + 1, len(beta)))

    # (0, 0) reduces to Legendre P_l(cos beta)
    if m == 0 and mp == 0:
        cos_b = np.cos(beta)
        for l in range(l_max + 1):
            d[l] = eval_legendre(l, cos_b)
        return d

    # Map (m, mp) to canonical form using symmetries.
    sign = 1.0
    m_eff, mp_eff = m, mp
    if abs(mp_eff) > abs(m_eff):
        sign *= (-1.0) ** (m_eff - mp_eff)
        m_eff, mp_eff = mp_eff, m_eff
    if m_eff < 0 or (m_eff + mp_eff) < 0:
        sign *= (-1.0) ** (m_eff - mp_eff)
        m_eff, mp_eff = -m_eff, -mp_eff
        if abs(mp_eff) > abs(m_eff):
            sign *= (-1.0) ** (m_eff - mp_eff)
            m_eff, mp_eff = mp_eff, m_eff

    l_min = max(abs(m_eff), abs(mp_eff))
    for l in range(l_min, l_max + 1):
        d[l] = sign * _d_general(l, m_eff, mp_eff, beta)
    return d


### Cell: `lensed_cls_TT_EE_TE` (CL05)


In [ ]:
from scipy.special import roots_legendre

def lensed_cls_TT_EE_TE(ell_arr, Cl_TT, Cl_EE, Cl_TE, Cl_phi_phi, n_theta=None):
    """Curved-sky lensed TT, EE, TE via the CL05 small-deflection expansion.
    Recommended: n_theta >= 2 * max(ell_arr) for sub-percent TT/EE/TE."""
    ell_arr = np.asarray(ell_arr, dtype=int)
    lmax    = int(ell_arr.max())
    if n_theta is None:
        n_theta = max(2 * lmax + 200, 500)
    ell_full = np.arange(lmax + 1)
    def _pad(C):
        return np.interp(ell_full, ell_arr, np.asarray(C), left=0.0, right=0.0)
    CTT, CEE, CTE, Cpp = _pad(Cl_TT), _pad(Cl_EE), _pad(Cl_TE), _pad(Cl_phi_phi)

    # Gauss-Legendre nodes and weights for the beta integral
    x, w  = roots_legendre(n_theta)
    beta  = np.arccos(x)

    # Wigner-d tables on the beta grid
    d_00  = wigner_d_array(lmax, 0,  0, beta)
    d_22  = wigner_d_array(lmax, 2,  2, beta)
    d_2m2 = wigner_d_array(lmax, 2, -2, beta)
    d_02  = wigner_d_array(lmax, 0,  2, beta)

    ell_w = (2 * ell_full + 1) / (4.0 * np.pi)
    Lpp   = ell_full * (ell_full + 1.0)

    # Total deflection variance and beta-dependent variance
    sigma2_alpha = np.sum((2 * ell_full + 1) * Lpp * Cpp / (4.0 * np.pi))
    C_gl_beta    = np.einsum('l,lb->b', (2*ell_full + 1) * Lpp * Cpp / (4.0*np.pi), d_00)
    sigma2_beta  = np.maximum(sigma2_alpha - C_gl_beta, 0.0)

    # Per-l Gaussian lensing modulation: exp(- L(L+1) sigma^2(beta) / 2)
    smoothing_per_l = np.exp(-0.5 * Lpp[:, None] * sigma2_beta[None, :])
    weight = ell_w[:, None] * smoothing_per_l

    # Lensed correlation functions
    xi_TT_lens = np.einsum('lb,lb->b', CTT[:, None] * weight, d_00)
    xi_p_lens  = np.einsum('lb,lb->b', CEE[:, None] * weight, d_22)
    xi_m_lens  = np.einsum('lb,lb->b', CEE[:, None] * weight, d_2m2)
    xi_TE_lens = -np.einsum('lb,lb->b', CTE[:, None] * weight, d_02)

    # Inverse transforms back to multipoles
    Cl_TT_lensed = 2.0 * np.pi * np.einsum('b,lb,b->l', xi_TT_lens, d_00, w)
    Cl_EE_lensed = (np.pi * np.einsum('b,lb,b->l', xi_p_lens, d_22, w)
                  + np.pi * np.einsum('b,lb,b->l', xi_m_lens, d_2m2, w))
    Cl_TE_lensed = -2.0 * np.pi * np.einsum('b,lb,b->l', xi_TE_lens, d_02, w)

    return {
        'ell': ell_arr,
        'Cl_TT_lensed': np.interp(ell_arr, ell_full, Cl_TT_lensed),
        'Cl_EE_lensed': np.interp(ell_arr, ell_full, Cl_EE_lensed),
        'Cl_TE_lensed': np.interp(ell_arr, ell_full, Cl_TE_lensed),
        'sigma2_alpha': sigma2_alpha,
    }


### Cell: `lensed_BB_from_EE_flat_sky`


In [ ]:
def lensed_BB_from_EE_flat_sky(ell_arr, Cl_EE_unl, Cl_phi_phi, n_l=200, n_phi=80):
    """Flat-sky BB from lensing of E, Hu-Okamoto / Lewis-Challinor formula."""
    ell_arr = np.asarray(ell_arr, dtype=int)
    lmax    = int(ell_arr.max())
    ell_full = np.arange(lmax + 1)
    CEE_full = np.interp(ell_full, ell_arr, np.asarray(Cl_EE_unl), left=0.0, right=0.0)
    Cpp_full = np.interp(ell_full, ell_arr, np.asarray(Cl_phi_phi), left=0.0, right=0.0)

    # Log-spaced l' grid; uniform alpha grid for the angular integral
    l_p   = np.geomspace(2.0, float(lmax), n_l)
    Cpp_p = np.interp(l_p, ell_full, Cpp_full)
    dlnlp = np.gradient(np.log(l_p))

    alpha  = np.linspace(0.0, 2.0 * np.pi, n_phi, endpoint=False)
    da     = 2.0 * np.pi / n_phi
    cos_a, sin_a = np.cos(alpha), np.sin(alpha)
    sin2_a = sin_a ** 2

    Cl_BB_lens = np.zeros(len(ell_full))
    for L in ell_full:
        if L < 2:
            continue
        Lm_sq = L*L + l_p[None, :]**2 - 2.0 * L * l_p[None, :] * cos_a[:, None]
        Lm_sq = np.maximum(Lm_sq, 0.25)
        Lm    = np.sqrt(Lm_sq)
        dot   = L * l_p[None, :] * cos_a[:, None] - l_p[None, :]**2
        Lmm   = L - l_p[None, :] * cos_a[:, None]
        sin2_2phi = 4.0 * l_p[None, :]**2 * sin2_a[:, None] * Lmm**2 / Lm_sq**2
        CEE_at = np.interp(Lm.ravel(), ell_full, CEE_full).reshape(Lm.shape)
        integrand = dot**2 * sin2_2phi * Cpp_p[None, :] * CEE_at
        Cl_BB_lens[L] = (1.0 / (2.0 * np.pi)**2) * np.sum(
            integrand * l_p[None, :]**2 * dlnlp[None, :] * da)

    return np.interp(ell_arr, ell_full, Cl_BB_lens)


### Cell: `lens_spectra`


In [ ]:
def lens_spectra(ell_arr, Cl_TT, Cl_EE, Cl_TE, Cl_phi_phi,
                 Cl_BB_unl=None, n_theta=None, n_l_BB=200, n_phi_BB=80):
    """Full lensed CMB spectra: TT, EE, BB, TE.
    Cl_BB_unl: optional unlensed BB (e.g. tensors from chapter 7)."""
    res = lensed_cls_TT_EE_TE(ell_arr, Cl_TT, Cl_EE, Cl_TE, Cl_phi_phi,
                              n_theta=n_theta)
    Cl_BB_from_EE = lensed_BB_from_EE_flat_sky(
        ell_arr, Cl_EE, Cl_phi_phi, n_l=n_l_BB, n_phi=n_phi_BB)
    if Cl_BB_unl is not None:
        # Smooth the input BB (e.g. tensor) by the same Gaussian lensing envelope
        sigma2 = res['sigma2_alpha']
        Lpp = ell_arr * (ell_arr + 1.0)
        Cl_BB_in_smoothed = np.asarray(Cl_BB_unl) * np.exp(-0.5 * Lpp * sigma2)
    else:
        Cl_BB_in_smoothed = np.zeros_like(ell_arr, dtype=float)
    res['Cl_BB_lensed']       = Cl_BB_from_EE + Cl_BB_in_smoothed
    res['Cl_BB_from_EE_lensing'] = Cl_BB_from_EE
    return res


### Cell: produce lensed scalar spectra


In [ ]:
ells_unl  = result['ells']
Cl_TT_unl = result['Dl_TT'] * 2*np.pi / (ells_unl*(ells_unl+1))
Cl_EE_unl = result['Dl_EE'] * 2*np.pi / (ells_unl*(ells_unl+1))
Cl_TE_unl = result['Dl_TE'] * 2*np.pi / (ells_unl*(ells_unl+1))
Cl_phi_interp = np.interp(ells_unl, phi['ell'], phi['Cl_phi_phi'])
lensed = lens_spectra(ells_unl, Cl_TT_unl, Cl_EE_unl, Cl_TE_unl, Cl_phi_interp)


### Cell: lensed vs unlensed


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8),
                         gridspec_kw={'height_ratios': [3, 1]})
for col, label, unl, lens in [
    (0, 'TT', Cl_TT_unl, lensed['Cl_TT_lensed']),
    (1, 'EE', Cl_EE_unl, lensed['Cl_EE_lensed']),
    (2, 'BB', np.zeros_like(Cl_TT_unl), lensed['Cl_BB_lensed'])]:
    top, bot = axes[0, col], axes[1, col]
    Dl_unl  = ells_unl*(ells_unl+1)/(2*np.pi) * unl
    Dl_lens = ells_unl*(ells_unl+1)/(2*np.pi) * lens
    if label == 'BB':
        top.loglog(ells_unl, Dl_lens, 'C3-', lw=1.4, label='lensed')
        top.set_ylim(1e-5, 1)
        bot.axis('off')
    else:
        top.loglog(ells_unl, Dl_unl, 'C0--', lw=1.2, label='unlensed')
        top.loglog(ells_unl, Dl_lens, 'C3-', lw=1.4, label='lensed')
        resid = 100*(Dl_lens - Dl_unl)/np.maximum(Dl_unl, 1e-30)
        bot.semilogx(ells_unl, resid, 'k-', lw=1.0)
        bot.axhline(0, color='gray', lw=0.5)
        bot.set_xlabel(r'$\ell$'); bot.set_ylabel('residual [%]')
        bot.set_xlim(2, 2500); bot.grid(True, which='both', alpha=0.3)
    top.set_title(label); top.set_xlim(2, 2500); top.legend()
    top.set_ylabel(rf'$\mathcal{{D}}_\ell^{{{label}}}\,[\mu K^2]$')
    top.grid(True, which='both', alpha=0.3)
plt.tight_layout(); plt.show()


# Chapter 7 — Tensors


### Cell: tensor truncation and indices


In [ ]:
LMAX_T_T = 10                                  # tensor temperature truncation
LMAX_E_T = 10
LMAX_B_T = 10

IX_TENS_H    = 0                               # h_T
IX_TENS_HDOT = 1                               # h_T dot
IX_TENS_T    = 2                               # Theta^T_2, _3, ...
IX_TENS_E    = IX_TENS_T + (LMAX_T_T - 1)      # E^T_2, _3, ...
IX_TENS_B    = IX_TENS_E + (LMAX_E_T - 1)      # B^T_2, _3, ...
NVAR_TENS    = IX_TENS_B + (LMAX_B_T - 1)


### Cell: `_tensor_kappa`


In [ ]:
def _tensor_kappa(ell):
    """Spin-2 angular-momentum coupling coefficient (Hu & White 1997)."""
    if ell < 2:
        return 0.0
    return np.sqrt((ell**2 - 4) * (ell**2 - 1)) / ell


### Cell: tensor Boltzmann RHS


In [ ]:
def _tensor_rhs(tau, y, k, bg_vec, sp_a_x, sp_a_c, sp_op_x, sp_op_c):
    """Tensor Boltzmann hierarchy in Hu-White conventions."""
    grhog, grhornomass, grhoc, grhob, grhov = bg_vec
    a = _cubic_eval(sp_a_x, sp_a_c, tau)
    a2 = a * a
    grho_a2 = grhog/a2 + grhornomass/a2 + grhoc/a + grhob/a + grhov*a2
    adotoa  = np.sqrt(grho_a2 / 3.0)
    opacity = max(_cubic_eval(sp_op_x, sp_op_c, tau), 1e-30)

    h_T    = y[IX_TENS_H]
    h_Tdot = y[IX_TENS_HDOT]

    Theta = np.zeros(LMAX_T_T + 2)
    E     = np.zeros(LMAX_E_T + 2)
    B     = np.zeros(LMAX_B_T + 2)
    for ell in range(2, LMAX_T_T + 1):
        Theta[ell] = y[IX_TENS_T + ell - 2]
    for ell in range(2, LMAX_E_T + 1):
        E[ell] = y[IX_TENS_E + ell - 2]
    for ell in range(2, LMAX_B_T + 1):
        B[ell] = y[IX_TENS_B + ell - 2]

    Pi_T = 0.1 * (Theta[2] - np.sqrt(6.0) * E[2])

    dy = np.zeros(NVAR_TENS)
    # GW wave equation
    dy[IX_TENS_H]    = h_Tdot
    dy[IX_TENS_HDOT] = -2.0 * adotoa * h_Tdot - k * k * h_T

    # Temperature hierarchy with metric source delta_{l,2} h_Tdot
    for ell in range(2, LMAX_T_T + 1):
        kappa_l   = _tensor_kappa(ell)
        kappa_lp1 = _tensor_kappa(ell + 1)
        T_lm1 = Theta[ell - 1] if ell > 2 else 0.0
        T_lp1 = Theta[ell + 1] if ell + 1 <= LMAX_T_T else 0.0
        if ell == 2:
            src = -h_Tdot - opacity * (Theta[ell] - Pi_T)
        elif ell == LMAX_T_T:
            src = -(ell + 1) / tau * Theta[ell] - opacity * Theta[ell]
            dy[IX_TENS_T + ell - 2] = k * T_lm1 + src
            continue
        else:
            src = -opacity * Theta[ell]
        dy[IX_TENS_T + ell - 2] = (k / (2.0 * ell + 1.0)) * (
            kappa_l * T_lm1 - kappa_lp1 * T_lp1) + src

    # E-B coupled hierarchies
    for ell in range(2, LMAX_E_T + 1):
        kappa_l = _tensor_kappa(ell); kappa_lp1 = _tensor_kappa(ell + 1)
        E_lm1 = E[ell - 1] if ell > 2 else 0.0
        E_lp1 = E[ell + 1] if ell + 1 <= LMAX_E_T else 0.0
        B_l   = B[ell]    if ell <= LMAX_B_T else 0.0
        if ell == 2:
            src_E = opacity * (-(E[ell] - np.sqrt(6.0) * Pi_T / 2.0))
        elif ell == LMAX_E_T:
            src_E = -(ell + 1) / tau * E[ell] - opacity * E[ell]
            dy[IX_TENS_E + ell - 2] = k * E_lm1 + src_E
            continue
        else:
            src_E = -opacity * E[ell]
        dy[IX_TENS_E + ell - 2] = (k / (2.0 * ell + 1.0)) * (
            kappa_l * E_lm1 - kappa_lp1 * E_lp1
        ) - 2.0 * k * B_l / (ell * (ell + 1.0)) + src_E

    for ell in range(2, LMAX_B_T + 1):
        kappa_l = _tensor_kappa(ell); kappa_lp1 = _tensor_kappa(ell + 1)
        B_lm1 = B[ell - 1] if ell > 2 else 0.0
        B_lp1 = B[ell + 1] if ell + 1 <= LMAX_B_T else 0.0
        E_l   = E[ell]    if ell <= LMAX_E_T else 0.0
        if ell == LMAX_B_T:
            src_B = -(ell + 1) / tau * B[ell] - opacity * B[ell]
            dy[IX_TENS_B + ell - 2] = k * B_lm1 + src_B
            continue
        dy[IX_TENS_B + ell - 2] = (k / (2.0 * ell + 1.0)) * (
            kappa_l * B_lm1 - kappa_lp1 * B_lp1
        ) + 2.0 * k * E_l / (ell * (ell + 1.0)) - opacity * B[ell]

    return dy


### Cell: `tensor_spectra`


In [ ]:
def tensor_spectra(params, bg, thermo, pgrid,
                   ells=None, r=0.05, n_t=None, N_k=200, N_tau=300):
    """Tensor-mode CMB angular power spectra.
    r   : tensor-to-scalar ratio at k_pivot
    n_t : tensor spectral index; None -> consistency relation n_t = -r/8
    """
    if n_t is None:
        n_t = -r / 8.0
    A_s = params['A_s']; A_t = r * A_s; k_pivot = params['k_pivot']

    k_arr   = np.geomspace(1e-5, 0.5, N_k)
    tau_min = max(1e-3, float(thermo['tau_arr'][0]))
    tau_max = float(bg['tau0']) * 0.9999
    tau_grid = np.geomspace(tau_min, tau_max, N_tau)
    vis    = thermo['visibility_interp'](tau_grid)
    exptau = thermo['exptau_interp'](tau_grid)

    if ells is None:
        ells = np.unique(np.concatenate([
            np.arange(2, 30), np.arange(30, 200, 5),
            np.arange(200, 1001, 25)]))
    ells = np.asarray(ells, dtype=int)

    histories = []
    for k in k_arr:
        tau_start = max(1e-4, min(1e-3 / k, tau_grid[0] * 0.5))
        y0 = np.zeros(NVAR_TENS); y0[IX_TENS_H] = 1.0
        sol = integrate.solve_ivp(
            _tensor_rhs, (tau_start, tau_grid[-1]), y0,
            args=(k, pgrid['bg_vec'], pgrid['sp_a_x'], pgrid['sp_a_c'],
                  pgrid['sp_op_x'], pgrid['sp_op_c']),
            t_eval=tau_grid, method='LSODA', rtol=1e-6, atol=1e-10)
        histories.append(sol.y if sol.success else None)

    chi = float(bg['tau0']) - tau_grid

    Cl_TT = np.zeros(len(ells)); Cl_EE = np.zeros(len(ells))
    Cl_BB = np.zeros(len(ells)); Cl_TE = np.zeros(len(ells))
    P_t   = A_t * (k_arr / k_pivot)**n_t

    for i_k, k in enumerate(k_arr):
        sol = histories[i_k]
        if sol is None:
            continue
        h_Tdot = sol[IX_TENS_HDOT]
        Theta_2 = sol[IX_TENS_T];    E_2 = sol[IX_TENS_E]
        Pi_T   = 0.1 * (Theta_2 - np.sqrt(6.0) * E_2)
        for i_ell, ell in enumerate(ells):
            ell_f       = float(ell)
            ell_factor  = np.sqrt((ell_f - 1) * ell_f * (ell_f + 1) * (ell_f + 2))
            x  = np.maximum(k * chi, 1e-30)
            jl = special.spherical_jn(int(ell), x)
            F_T = ell_factor * jl / x**2

            # Temperature source: ISW-like -h_Tdot e^{-kappa} + visibility * Pi_T
            S_T = -h_Tdot * exptau + vis * Pi_T
            S_E = vis * Pi_T * np.sqrt(6.0) / 4.0
            S_B = vis * Pi_T * np.sqrt(6.0) / 4.0
            Delta_T = np.trapz(S_T * F_T, tau_grid)
            Delta_E = np.trapz(S_E * F_T, tau_grid)
            Delta_B = np.trapz(S_B * F_T, tau_grid)

            weight = 4.0 * np.pi * P_t[i_k]
            dlnk   = np.log(k_arr[1] / k_arr[0]) if i_k > 0 else 0.0
            Cl_TT[i_ell] += weight * Delta_T**2 * dlnk
            Cl_EE[i_ell] += weight * Delta_E**2 * dlnk
            Cl_BB[i_ell] += weight * Delta_B**2 * dlnk
            Cl_TE[i_ell] += weight * Delta_T * Delta_E * dlnk

    T0_muK2 = (params['T_cmb'] * 1e6)**2
    Cl_TT *= T0_muK2; Cl_EE *= T0_muK2; Cl_BB *= T0_muK2; Cl_TE *= T0_muK2
    Dl = lambda C: ells * (ells + 1) / (2 * np.pi) * C
    return {'ell': ells,
            'Cl_TT_tensor': Cl_TT, 'Cl_EE_tensor': Cl_EE,
            'Cl_BB_tensor': Cl_BB, 'Cl_TE_tensor': Cl_TE,
            'Dl_TT_tensor': Dl(Cl_TT), 'Dl_EE_tensor': Dl(Cl_EE),
            'Dl_BB_tensor': Dl(Cl_BB), 'Dl_TE_tensor': Dl(Cl_TE)}


### Cell: tensor BB at several values of $r$


In [ ]:
pgrid = init_perturbation_grid(bg, th)
results_t = {r: tensor_spectra(params, bg, th, pgrid, r=r)
             for r in (0.05, 0.01, 0.001)}

fig, ax = plt.subplots(figsize=(7.5, 5))
for r, color in zip([0.05, 0.01, 0.001], ['C3', 'C4', 'C0']):
    Dl_BB_t = results_t[r]['Dl_BB_tensor']
    ell_t   = results_t[r]['ell']
    ax.loglog(ell_t, Dl_BB_t, color=color, lw=1.5, label=f'tensor BB, $r={r}$')
# Overlay lensing BB
ax.loglog(ells_unl, lensed['Cl_BB_lensed'] * ells_unl * (ells_unl + 1) / (2 * np.pi),
          'k--', lw=1.4, label='lensing BB ($r=0$)')
ax.set_xlabel(r'$\ell$'); ax.set_ylabel(r'$\mathcal{D}_\ell^{BB}\,[\mu K^2]$')
ax.set_title('Tensor BB vs lensing BB')
ax.set_xlim(2, 2500); ax.set_ylim(1e-7, 1e-1)
ax.legend(); ax.grid(True, which='both', alpha=0.3)
plt.show()


### Cell: total spectra and CAMB validation


In [ ]:
import camb
pars = camb.set_params(H0=100*params['h'],
                       ombh2=params['omega_b_h2'],
                       omch2=params['omega_c_h2'],
                       As=params['A_s'], ns=params['n_s'],
                       tau=params['tau_reion'], r=0.05)
pars.set_for_lmax(2500, lens_potential_accuracy=2)
pars.WantTensors = True
camb_powers = camb.get_results(pars).get_cmb_power_spectra(
    pars, CMB_unit='muK', raw_cl=False)
Dl_camb_lens   = camb_powers['lensed_scalar']
Dl_camb_tens   = camb_powers['tensor']

# Tutorial total: lensed scalar + tensor
r_target = 0.05
tens_r = results_t[r_target]
Dl_BB_t_interp = np.interp(ells_unl, tens_r['ell'], tens_r['Dl_BB_tensor'])
Dl_TT_t_interp = np.interp(ells_unl, tens_r['ell'], tens_r['Dl_TT_tensor'])
Dl_EE_t_interp = np.interp(ells_unl, tens_r['ell'], tens_r['Dl_EE_tensor'])

Dl_TT_tot = (ells_unl*(ells_unl+1)/(2*np.pi)) * lensed['Cl_TT_lensed'] + Dl_TT_t_interp
Dl_EE_tot = (ells_unl*(ells_unl+1)/(2*np.pi)) * lensed['Cl_EE_lensed'] + Dl_EE_t_interp
Dl_BB_tot = (ells_unl*(ells_unl+1)/(2*np.pi)) * lensed['Cl_BB_lensed'] + Dl_BB_t_interp


### Cell: final 4-panel validation plot


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
ells_camb = np.arange(Dl_camb_lens.shape[0])

for ax, label, idx, col in [
    (axes[0,0], 'TT', 0, Dl_TT_tot),
    (axes[0,1], 'EE', 1, Dl_EE_tot),
    (axes[1,0], 'BB lensed (scalar)', 2, (ells_unl*(ells_unl+1)/(2*np.pi))*lensed['Cl_BB_lensed']),
    (axes[1,1], 'BB tensor (r=0.05)', 2, Dl_BB_t_interp),
    ]:
    if 'tensor' in label:
        camb_y = Dl_camb_tens[:, 2]
    elif 'lensed' in label:
        # lensing BB from CAMB = lensed_scalar BB
        camb_y = Dl_camb_lens[:, 2]
    else:
        camb_y = Dl_camb_lens[:, idx]
    ax.plot(ells_unl, col, 'k-', lw=1.4, label='tutorial')
    ax.plot(ells_camb, camb_y, 'C3--', lw=1.0, label='CAMB')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlim(2, 2500)
    ax.set_xlabel(r'$\ell$')
    ax.set_ylabel(rf'$\mathcal{{D}}_\ell\,[\mu K^2]$')
    ax.set_title(label); ax.legend(); ax.grid(True, which='both', alpha=0.3)
plt.tight_layout(); plt.show()
